<a href="https://colab.research.google.com/github/Sunil-Paliwal/ai-enabled-drug-discovery-and-synthesis/blob/main/02_structure_validation_and_alerts/structure_validation_and_alerts.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


<table><tr><td style="background:#fff8e6; border-left:5px solid #d68910; padding:10px 16px; font-family:system-ui,sans-serif; font-size:14px;">
<b>&#9654;&nbsp; To run this notebook, click the <i>Open in Colab</i> badge above.</b><br>
GitHub shows a read-only preview &mdash; the code will not run on this page. Colab runs it in your browser for free; all you need is a Google account, and nothing is installed on your computer.<br>
<span style="color:#555;">Once it opens in Colab, run the first two cells (install, then setup) before anything else.</span>
</td></tr></table>


### Structure Validation and Alerts

**Before you ask whether a molecule is active, ask whether it exists.**

<div style="font-size:13px; color:#333; border-left:3px solid #2c5c40;
            padding:8px 16px; margin:12px 0; background:#f7f9f7;">
<b>Author:</b> Dr. Sunil Paliwal, Department of Chemistry and Chemical Biology,
Stevens Institute of Technology<br>
<b>Module:</b> 02 of <a href="https://github.com/Sunil-Paliwal/ai-enabled-drug-discovery-and-synthesis">AI-Enabled
Drug Discovery and Synthesis</a> &nbsp;&middot;&nbsp;
<b>Version:</b> 1.0 &nbsp;&middot;&nbsp; <b>Licence:</b> MIT<br>
<b>Status:</b> Developed for teaching and research in drug discovery and medicinal
chemistry.<br>
<b>Cite as:</b> Paliwal, S. <i>Structure Validation and Alerts.</i> GitHub, 2026.
github.com/Sunil-Paliwal/ai-enabled-drug-discovery-and-synthesis<br>
<span style="font-size:12px; color:#666;">Working from a saved copy? The link above
always has the current version.</span>
</div>

---

## Background

When you design a molecule, two different things can be wrong with it, and they fail in
two different ways.

The first is that **it is not a molecule at all**. A ring drawn as aromatic with a CH<sub>2</sub>
sitting in it. A carbon with five bonds. An unclosed ring. These are not chemistry mistakes
so much as drawing mistakes, and software does not politely flag them &mdash; RDKit refuses to
build the molecule and throws an error written for programmers, like
`Can't kekulize mol. Unkekulized atoms: 0 1 2 4 5`. If your pipeline does not catch that,
the molecule silently disappears from your dataset, or your notebook crashes, and you learn
nothing about why.

The second is that it **is** a valid molecule but it **will not survive**, or it will react
with everything. An aldehyde. An acyl chloride. A gem-diol, which is just the hydrate of a
carbonyl and loses water the moment you put it in a flask. An enol drawn where the ketone is
the form that actually exists. These parse perfectly. They are chemically meaningful and
chemically doomed.

This notebook checks both, in that order, because the second check cannot run until the
first one passes.

---

## What it does

**Stage 1 &mdash; structural validity.** Parses your SMILES *without* sanitising it, asks RDKit
what is wrong, maps each problem back to a specific atom, and translates the error into a
sentence you can act on. You get the molecule drawn with the offending atom circled in red,
and a suggested fix. Nothing downstream runs until this passes.

**Stage 1.5 &mdash; is the drawing complete?** A structure can parse, sanitise, and trip no
alert, and still not describe a single compound. A 1,4-disubstituted cyclohexane drawn with
no stereochemistry is two molecules with two different shapes. Unspecified stereocentres and
undefined double-bond geometry are reported here, separately from the alerts, because
"underspecified" is a different problem from "unstable".

**Stage 2 &mdash; stability and reactivity.** Runs the molecule against three published
substructure catalogues (PAINS, Brenk, NIH) plus a hand-written set of patterns for the
things that are *unstable as drawn* &mdash; hemiacetals, gem-diols, enols, anhydrides,
carbamic acids, aminals, peroxides &mdash; which the published lists do not cover, because
they were built to triage screening libraries, not to catch design errors. Every hit comes
with the chemistry behind it, a suggested replacement, and where relevant a group name to
look up in the **Bioisostere Explorer** notebook.

**Mutagenicity, with the mechanism.** Almost every classic mutagenicity toxicophore
(Kazius 2005; Benigni&ndash;Bossa) already fires through those three catalogues &mdash; but they
return a bare name like `nitro_group` and an explanation that says only that it is on a list.
Those hits are relabelled `MUTAGENIC` and carry the actual mechanism instead: a nitroaromatic
is reduced to the nitroso and then the N-hydroxy compound; an aromatic amine is oxidised to
the same N-hydroxy compound and then to a nitrenium ion that alkylates guanine. **Severity is
capped at MEDIUM deliberately** &mdash; see the caution below.

**Considered and not flagged.** Silence looks like an oversight. If your molecule contains a
ketone, an ester, a phenol, an aryl halide, a carboxylic acid, a basic amine or a nitrile, the
report says so and explains why each is *not* being treated as an alert. These do not count
towards the verdict and never turn a green molecule amber &mdash; they are there so you know
the tool looked rather than missed.

**A colour-coded verdict.** Every molecule gets one of four banners &mdash; **dark red** (not a
valid molecule &mdash; the most severe outcome), **red** (a high-severity alert), **amber**
(worth justifying), **green** (no alerts matched). Within the drawing, each flagged atom is coloured by the worst severity
that hit it, so one picture can show a red aldehyde and an amber catechol at the same time.
Note what green does *not* say: it is the absence of a flag, not evidence of a good molecule.

**A reality check.** Eleven approved drugs run through the same filters at the end. Most of them
trip something. An alert is a reason to justify a design choice, not a reason to delete a
molecule &mdash; and where the flagged group *is* the mechanism, as with the &beta;-lactam in
penicillin or the Michael acceptor in afatinib, removing it would remove the drug.

**Batch mode.** The same two stages over a file of thousands of SMILES, reporting not just
*how many* were valid but *which failure modes dominated* &mdash; which is what you want when
you are auditing the output of a generative model.

<div style="border:2px solid #3f7d58; border-radius:10px; padding:4px 22px 18px 22px;
            background:#f4faf6; color:#1b1b1b; font-size:15px;">

<h2 style="color:#2c5c40; margin-bottom:2px;"><b>How to use this notebook</b></h2>

<p>Look for the <b>green bars</b>: they mark <b>&#9733; Step 1</b> through
<b>&#9733; Step 5</b>, which is all you have to follow. Only <b>three</b> cells need
editing, and those carry an orange <b>you edit this one</b> tag. Everything else you
simply run.</p>

<table style="width:100%; border-collapse:collapse; background:#ffffff; color:#1b1b1b;">
<tr style="background:#e4efe8;">
  <th style="text-align:left; padding:7px 10px; width:4.5em;">Cell</th>
  <th style="text-align:left; padding:7px 10px;">What it does</th>
  <th style="text-align:left; padding:7px 10px; width:7em;">You edit it?</th>
</tr>

<tr><td colspan="3" style="padding:7px 10px; background:#f0f5f2;"><b>Step 1 &mdash; Setup</b></td></tr>
<tr><td style="padding:7px 10px;"><b>1</b></td>
    <td style="padding:7px 10px;">Installs RDKit. Takes about a minute.</td>
    <td style="padding:7px 10px;">no</td></tr>
<tr><td style="padding:7px 10px;"><b>2</b></td>
    <td style="padding:7px 10px;">Loads the engine: 37 hand-written patterns plus the PAINS, Brenk and NIH catalogues.</td>
    <td style="padding:7px 10px;">no</td></tr>

<tr><td colspan="3" style="padding:7px 10px; background:#f0f5f2;"><b>Step 2 &mdash; Check your molecule</b></td></tr>
<tr><td style="padding:7px 10px;"><b>3</b></td>
    <td style="padding:7px 10px;">Paste one SMILES. Get both stages, the highlighted drawing, and the reasons.</td>
    <td style="padding:7px 10px;"><b>yes</b></td></tr>

<tr><td colspan="3" style="padding:7px 10px; background:#f0f5f2;"><b>Step 3 &mdash; Reality check</b></td></tr>
<tr><td style="padding:7px 10px;"><b>4</b></td>
    <td style="padding:7px 10px;">Approved drugs through the same filter. Read this before you trust any alert.</td>
    <td style="padding:7px 10px;">no</td></tr>

<tr><td colspan="3" style="padding:7px 10px; background:#f0f5f2;"><b>Step 4 &mdash; Many molecules at once</b></td></tr>
<tr><td style="padding:7px 10px;"><b>5</b></td>
    <td style="padding:7px 10px;">Batch mode: validity rate and the failure-mode breakdown for a whole list.</td>
    <td style="padding:7px 10px;"><b>yes</b></td></tr>

<tr><td colspan="3" style="padding:7px 10px; background:#f0f5f2;"><b>Step 5 &mdash; Keep the result</b></td></tr>
<tr><td style="padding:7px 10px;"><b>6</b></td>
    <td style="padding:7px 10px;">Writes a PDF you can hand in, and an HTML version for reading in a browser.</td>
    <td style="padding:7px 10px;">no</td></tr>
<tr><td style="padding:7px 10px;"><b>7</b></td>
    <td style="padding:7px 10px;">Add your own alert pattern.</td>
    <td style="padding:7px 10px;"><b>yes</b></td></tr>
</table>

<p style="margin-top:14px;"><b>Where to get a SMILES.</b> Draw your molecule in ChemDraw,
MarvinSketch, or at <a href="https://pubchem.ncbi.nlm.nih.gov//edit3/index.html">PubChem
Sketcher</a>, then copy the SMILES it exports. <b>Do not type it by hand</b> and do not work
from a picture &mdash; a drawing cannot tell you whether a bond is where you think it is, and
that ambiguity is exactly what this notebook exists to remove.</p>

</div>

<div style="background:#2c5c40; color:#ffffff; padding:12px 22px; border-radius:8px;
            margin-top:28px;">
<h2 style="color:#ffffff; margin:0; font-size:30px;">&#9733; STEP 1 &mdash; Setup</h2>
</div>

**Cell 1** installs RDKit, **Cell 2** loads the engine.
**Run both every time you open the notebook.** Colab starts with a fresh machine each
session, so nothing from last time is still loaded.

In [ ]:
# ===== CELL 1 =====
!pip install rdkit pandas reportlab --quiet
print('installing...')

In [ ]:
# ===== CELL 2 =====
"""
Structure Validation and Alerts - engine
Stage 1: structural validity   Stage 2: stability & reactivity alerts
"""

# ----------------------------------------------------------------------
#  VERSION
# ----------------------------------------------------------------------
#  Bump this whenever the engine changes. Students who clicked "Copy to
#  Drive" keep that copy forever and never see an update, so when someone
#  reports odd behaviour the first question is which version they are on.
#  It is printed when Cell 2 runs and stamped into every report.
VERSION = "1.0"
VERSION_DATE = "2026-11-08"
TOOL_AUTHOR = "Sunil Paliwal"
TOOL_URL = "https://github.com/Sunil-Paliwal/ai-enabled-drug-discovery-and-synthesis"

import os, sys, re, textwrap
import pandas as pd
from rdkit import Chem, RDLogger
from rdkit.Chem import Draw, AllChem, Descriptors, rdMolDescriptors
from rdkit.Chem.Draw import rdMolDraw2D
from rdkit.Chem import FilterCatalog
from rdkit.Chem.FilterCatalog import FilterCatalogParams
from IPython.display import display, HTML, SVG

RDLogger.DisableLog("rdApp.*")          # we report problems ourselves

# ----------------------------------------------------------------------
#  SEVERITY
# ----------------------------------------------------------------------
#  BLOCKER  this is not a molecule; nothing downstream can run
#  HIGH     will not survive as drawn, or is a hard covalent liability
#  MEDIUM   a real alert, but defensible with a reason
#  NOTE     context dependent; shown so you can think about it
SEV_ORDER = {"BLOCKER": 0, "HIGH": 1, "MEDIUM": 2, "NOTE": 3}
SEV_COLOUR = {"BLOCKER": "#8b0000", "HIGH": "#c0392b",
              "MEDIUM": "#d68910", "NOTE": "#2c5c40"}

# ----------------------------------------------------------------------
#  CUSTOM ALERTS
# ----------------------------------------------------------------------
#  Every entry:
#    name      short label shown in the table
#    smarts    the pattern
#    cat       UNSTABLE / REACTIVE / STRAIN
#    sev       HIGH / MEDIUM / NOTE
#    why       the chemistry, in one sentence a student can act on
#    fix       what to draw instead, or "" when there is no generic answer
#    iso       group name to look up in the Bioisostere Explorer, or ""
ALERTS = [

    # ---------- unstable as drawn: these fall apart in water -----------
    dict(name="gem-diol (hydrate)", smarts="[CX4]([OX2H])([OX2H])",
         cat="UNSTABLE", sev="HIGH",
         why="two OH groups on one carbon; this is the hydrate of a carbonyl "
             "and it loses water to give back the aldehyde or ketone",
         fix="draw the carbonyl instead: C=O", iso=""),

    dict(name="hemiacetal", smarts="[CX4;H1,H2]([OX2H])[OX2][#6]", core=(0, 1, 2),
         cat="UNSTABLE", sev="HIGH",
         why="OH and OR on the same carbon; opens to the aldehyde plus alcohol "
             "unless it is locked in a sugar-like ring",
         fix="draw the acetal (two OR) or the open aldehyde", iso=""),

    dict(name="hemiketal", smarts="[CX4;H0]([OX2H])([OX2][#6])", core=(0, 1, 2),
         cat="UNSTABLE", sev="HIGH",
         why="OH and OR on the same quaternary carbon; reverts to the ketone "
             "plus alcohol unless held in a ring",
         fix="draw the ketal or the open ketone", iso=""),

    dict(name="hemiaminal", smarts="[NX3][CX4;H1,H2][OX2H]",
         cat="UNSTABLE", sev="HIGH",
         why="OH and N on the same carbon; collapses to the iminium or back to "
             "the carbonyl plus amine",
         fix="draw the amide, or the amine and carbonyl separately", iso=""),

    dict(name="aminal (1,1-diamine)", smarts="[NX3][CX4;H1,H2][NX3]",
         cat="UNSTABLE", sev="HIGH",
         why="two nitrogens on one carbon; hydrolyses readily in water",
         fix="draw an amidine, a guanidine, or move one N away", iso="amidine"),

    dict(name="enol", smarts="[OX2H][CX3]=[CX3]",
         cat="UNSTABLE", sev="HIGH",
         why="OH on an sp2 carbon of a C=C; almost always tautomerises to the "
             "ketone or aldehyde, which is the form that actually exists",
         fix="draw the keto tautomer", iso=""),

    dict(name="gem halohydrin", smarts="[CX4]([OX2H])[F,Cl,Br,I]",
         cat="UNSTABLE", sev="HIGH",
         why="OH and halogen on the same carbon; eliminates HX to give the "
             "carbonyl",
         fix="draw the carbonyl", iso=""),

    dict(name="carbamic acid", smarts="[NX3][CX3](=[OX1])[OX2H]",
         cat="UNSTABLE", sev="HIGH",
         why="free N-CO-OH decarboxylates to the amine and CO2",
         fix="draw the carbamate ester, the urea, or the free amine", iso=""),

    dict(name="peroxide", smarts="[OX2][OX2]",
         cat="UNSTABLE", sev="HIGH",
         why="O-O single bond; thermally and photochemically labile, and a "
             "genuine handling hazard",
         fix="remove one oxygen or use an ether", iso=""),

    dict(name="N-N-N chain", smarts="[NX3][NX3][NX3]",
         cat="UNSTABLE", sev="HIGH",
         why="three consecutive sp3 nitrogens; energetic and prone to "
             "decomposition",
         fix="break up the chain with a carbon", iso=""),

    dict(name="orthoester", smarts="[CX4]([OX2][#6])([OX2][#6])[OX2][#6]", core=(0, 1, 3, 5),
         cat="UNSTABLE", sev="MEDIUM",
         why="three OR groups on one carbon; hydrolyses under mild acid",
         fix="draw the ester", iso=""),

    dict(name="acyl halide", smarts="[CX3](=[OX1])[F,Cl,Br,I]",
         cat="UNSTABLE", sev="HIGH",
         why="hydrolyses in seconds to minutes in water; this is a reagent, "
             "not a compound that reaches an assay",
         fix="draw the amide, ester or acid it would become", iso=""),

    dict(name="anhydride", smarts="[CX3](=[OX1])[OX2][CX3](=[OX1])",
         cat="UNSTABLE", sev="HIGH",
         why="hydrolyses rapidly in water to two acids",
         fix="draw the acid or the amide", iso=""),

    dict(name="imine / Schiff base", smarts="[CX3;H1,H0]=[NX2;H1][#6]", core=(0, 1),
         cat="UNSTABLE", sev="MEDIUM",
         why="an unactivated C=N hydrolyses back to the carbonyl and amine at "
             "physiological pH unless it is in a ring or conjugated",
         fix="reduce it to the amine, or use an amidine or aromatic N "
             "heterocycle", iso="amidine"),

    # ---------- activated ketones --------------------------------------
    #  A PLAIN KETONE IS NOT LISTED HERE, AND THAT IS DELIBERATE.
    #  Ketone electrophilicity spans orders of magnitude and the
    #  substituents decide where on that range a given one sits. In water
    #  formaldehyde is >99.9% hydrated, acetaldehyde roughly half, acetone
    #  a few tenths of a percent - same C=O, three orders of magnitude.
    #  Haloperidol, ketamine, ketoprofen and nabumetone all carry plain
    #  ketones. Put electron-withdrawing groups next to the carbonyl,
    #  though, and it becomes a genuine warhead. Those are the ones below.
    dict(name="alpha-ketoamide", smarts="[#6][CX3](=[OX1])[CX3](=[OX1])[NX3]", core=(1, 2, 3, 4, 5),
         cat="REACTIVE", sev="HIGH",
         why="two carbonyls in a row make this ketone strongly electrophilic; "
             "the catalytic serine or cysteine of a protease adds to it to "
             "give a stable hemiketal, which is exactly how telaprevir and "
             "boceprevir inhibit HCV protease",
         fix="separate the two carbonyls, or keep it if a reversible covalent "
             "mechanism is the plan", iso=""),

    dict(name="alpha-ketoester", smarts="[#6][CX3](=[OX1])[CX3](=[OX1])[OX2][#6]", core=(1, 2, 3, 4, 5),
         cat="REACTIVE", sev="HIGH",
         why="the ester pulls electron density out of the neighbouring "
             "ketone, leaving a carbonyl electrophilic enough to be attacked "
             "by serine and by lysine",
         fix="separate the two carbonyls, or use the simple ester", iso=""),

    dict(name="1,2-diketone", smarts="[#6][CX3](=[OX1])[CX3](=[OX1])[#6]", core=(1, 2, 3, 4),
         cat="REACTIVE", sev="HIGH",
         why="adjacent carbonyls activate each other; these hydrate readily, "
             "condense with lysine and arginine, and redox cycle",
         fix="move one carbonyl, or reduce it", iso=""),

    dict(name="trifluoromethyl ketone", smarts="[CX3](=[OX1])[CX4](F)(F)F",
         cat="REACTIVE", sev="HIGH",
         why="the CF3 group makes the carbonyl so electrophilic that the "
             "compound exists largely as its hydrate and forms tight "
             "hemiketals with serine hydrolases; a deliberate reversible "
             "covalent warhead",
         fix="use the methyl ketone, or keep it if covalent inhibition is "
             "the plan", iso=""),

    # ---------- reactive: covalent liability ---------------------------
    dict(name="aldehyde", smarts="[CX3H1](=O)[#6]", core=(0, 1),
         cat="REACTIVE", sev="HIGH",
         why="a hard electrophile; reacts with lysine and cysteine, is "
             "oxidised fast to the acid, and is a frequent source of "
             "non-specific assay activity",
         fix="replace with nitrile, oxazole, amide or a masked equivalent",
         iso="aldehyde"),

    dict(name="Michael acceptor", smarts="[CX3]=[CX3][CX3]=[OX1]",
         cat="REACTIVE", sev="HIGH",
         why="an alpha,beta-unsaturated carbonyl; accepts thiols, so it reacts "
             "with glutathione and with cysteines across the proteome",
         fix="saturate the double bond, or keep it only if a covalent "
             "mechanism is the plan", iso=""),

    dict(name="epoxide", smarts="[CX4]1[OX2][CX4]1",
         cat="REACTIVE", sev="HIGH",
         why="strained three-membered ring; opened by any nucleophile, "
             "including DNA, which is why most are mutagenic",
         fix="open it to the diol or the amino alcohol", iso=""),

    dict(name="aziridine", smarts="[CX4]1[NX3][CX4]1",
         cat="REACTIVE", sev="HIGH",
         why="strained three-membered ring; a DNA alkylator",
         fix="open the ring or use azetidine or pyrrolidine", iso=""),

    dict(name="isocyanate", smarts="[NX2]=[CX2]=[OX1]",
         cat="REACTIVE", sev="HIGH",
         why="acylates amines and water on contact; a reagent for making "
             "ureas, not a compound",
         fix="draw the urea or carbamate product", iso=""),

    dict(name="isothiocyanate", smarts="[NX2]=[CX2]=[SX1]",
         cat="REACTIVE", sev="HIGH",
         why="reacts with amines and thiols; also a potent sensitiser",
         fix="draw the thiourea product", iso=""),

    #  Cl, Br and I only. Fluoride is a poor leaving group in substitution,
    #  so an alpha-fluoro ketone is not an alkylating agent - it is an
    #  activated carbonyl, which is a different alert (see the
    #  trifluoromethyl ketone above).
    dict(name="alpha-halo carbonyl", smarts="[CX3](=[OX1])[CX4][Cl,Br,I]",
         cat="REACTIVE", sev="HIGH",
         why="a strong alkylating agent; the classic non-selective "
             "cysteine reagent",
         fix="remove the halogen", iso=""),

    dict(name="sulfonyl halide", smarts="[SX4](=[OX1])(=[OX1])[F,Cl,Br,I]",
         cat="REACTIVE", sev="HIGH",
         why="sulfonylates amines and hydroxyls; a reagent, and sulfonyl "
             "fluorides are deliberate covalent warheads",
         fix="draw the sulfonamide", iso="sulfonamide"),

    dict(name="nitrogen mustard", smarts="[NX3]([CX4][CX4][Cl,Br,I])",
         cat="REACTIVE", sev="HIGH",
         why="cyclises to an aziridinium and alkylates DNA; cytotoxic by "
             "design in older chemotherapeutics",
         fix="remove the halide", iso=""),

    dict(name="acyl cyanide", smarts="[CX3](=[OX1])[CX2]#[NX1]",
         cat="REACTIVE", sev="HIGH",
         why="extremely electrophilic carbonyl; hydrolyses and acylates "
             "nucleophiles",
         fix="draw the amide or the nitrile alone", iso=""),

    dict(name="quinone", smarts="[OX1]=[CX3]1[CX3]=[CX3][CX3](=[OX1])[CX3]=[CX3]1",
         cat="REACTIVE", sev="HIGH",
         why="redox cycles to make reactive oxygen species and is a double "
             "Michael acceptor; the single most common PAINS motif",
         fix="use the non-oxidisable aromatic ring", iso=""),

    dict(name="thiol", smarts="[SX2H]",
         cat="REACTIVE", sev="MEDIUM",
         why="oxidises to the disulfide in air and in assay buffer, so the "
             "compound you tested is often not the one you drew",
         fix="use the thioether, or accept it and run under reducing "
             "conditions", iso=""),

    dict(name="hydrazine", smarts="[NX3;H1,H2][NX3;H1,H2]",
         cat="REACTIVE", sev="MEDIUM",
         why="N-N with free NH; oxidises easily and carries a well known "
             "genotoxicity and hepatotoxicity burden",
         fix="use an amine, amide or N-heterocycle", iso=""),

    dict(name="N-aryl hydroxylamine", smarts="[OX2H][NX3][c]",
         cat="MUTAGENIC", sev="MEDIUM",
         why="the N-hydroxy aromatic amine is the LAST intermediate before "
             "the nitrenium ion that alkylates guanine; nitroaromatics are "
             "reduced to it and aromatic amines are oxidised to it, so "
             "drawing it directly skips the activation step entirely",
         fix="use the plain aromatic amine, or cap the nitrogen as an amide",
         iso=""),

    dict(name="aromatic nitroso", smarts="[OX1]=[NX2][c]",
         cat="MUTAGENIC", sev="MEDIUM",
         why="an aryl nitroso group is the reactive intermediate that "
             "nitroaromatics and aromatic amines are converted INTO in the "
             "body; drawn directly it needs no activation, and it reacts with "
             "DNA bases and with protein thiols",
         fix="use the amine or remove the nitrogen; if you need the nitro "
             "group, understand that it is reduced to this",
         iso=""),

    dict(name="azo", smarts="[#6][NX2]=[NX2][#6]", core=(1, 2),
         cat="REACTIVE", sev="MEDIUM",
         why="reduced by gut bacteria to two anilines, which brings the "
             "aniline liabilities with it",
         fix="replace with an amide or a 1,2,3-triazole", iso=""),

    dict(name="activated aryl halide (SNAr)",
         smarts="[F,Cl][c;$(c1c([N+](=O)[O-])cccc1),$(c1cc([N+](=O)[O-])ccc1),$(c1c(C#N)cccc1),$(c1nccnc1),$(c1ncccn1)]",
         cat="REACTIVE", sev="MEDIUM",
         why="a halogen on an electron-poor ring is displaced by thiols; "
             "glutathione does this in the liver",
         fix="move the halogen, or remove the activating group", iso=""),

    dict(name="beta-lactam", smarts="O=C1[NX3][CX4][CX4]1",
         cat="REACTIVE", sev="NOTE",
         why="strained, acylating ring; this is a deliberate warhead in the "
             "penicillins, so it is only a problem if you did not mean it",
         fix="", iso=""),

    dict(name="boronic acid", smarts="[BX3]([OX2H])[OX2H]",
         cat="REACTIVE", sev="NOTE",
         why="forms reversible covalent bonds with serine and with diols; "
             "intentional in bortezomib, accidental if it is left over from "
             "a coupling",
         fix="", iso=""),
]

# ----------------------------------------------------------------------
#  USER-ADDED ALERTS
# ----------------------------------------------------------------------
def add_alert(name, smarts, cat="UNSTABLE", sev="MEDIUM", why="", fix="",
              iso="", core=None):
    """
    Add your own pattern. cat: UNSTABLE / REACTIVE / STRAIN / MUTAGENIC.
    core: which atoms OF THE SMARTS to colour, by position, e.g. (0, 1).
          Leave it None to colour the whole match. Use it when the
          pattern needs a context atom to match correctly but that atom
          is not itself part of the problem - the [#6] in an aldehyde
          pattern anchors it to a carbon, but it is just a neighbour.
    """
    patt = Chem.MolFromSmarts(smarts)
    if patt is None:
        print(f"  ! '{smarts}' is not a valid SMARTS - nothing was added")
        return
    if cat not in ("UNSTABLE", "REACTIVE", "STRAIN", "MUTAGENIC"):
        print(f"  ! cat must be UNSTABLE, REACTIVE, STRAIN or MUTAGENIC "
              f"- got '{cat}'")
        return
    if sev not in SEV_ORDER:
        print(f"  ! sev must be BLOCKER, HIGH, MEDIUM or NOTE - got '{sev}'")
        return
    ALERTS[:] = [a for a in ALERTS if a["name"] != name]
    ALERTS.append(dict(name=name, smarts=smarts, cat=cat, sev=sev,
                       why=why, fix=fix, iso=iso, core=core))
    _compile_alerts()
    print(f"  added '{name}'  ({cat}, {sev})  - {len(ALERTS)} patterns now active")


_COMPILED = []
def _compile_alerts():
    _COMPILED.clear()
    for a in ALERTS:
        p = Chem.MolFromSmarts(a["smarts"])
        if p is not None:
            _COMPILED.append((a, p))
_compile_alerts()


# ----------------------------------------------------------------------
#  MUTAGENICITY ANNOTATIONS
# ----------------------------------------------------------------------
#  Almost every classic mutagenicity toxicophore (Kazius 2005;
#  Benigni-Bossa) ALREADY fires through the Brenk, NIH and PAINS
#  catalogues. What those catalogues return is a bare name -
#  "Brenk: nitro_group" - and an explanation that says only that it is on
#  a list. That teaches nothing. This table attaches the actual mechanism
#  to hits the engine was already producing.
#
#  READ THIS BEFORE TRUSTING ANY OF IT: a mutagenicity alert is a
#  different KIND of claim from the rest of Stage 2. "This hemiacetal
#  hydrolyses" is chemistry you can predict from structure. "This is
#  mutagenic" is a prediction about the outcome of an assay in a living
#  system, and these rule bases carry substantial false-positive rates.
#  Metronidazole is Ames-positive and has been a front-line antibiotic
#  for decades. Severity here is capped at MEDIUM on purpose, and real
#  assessment runs through Toxtree's Benigni-Bossa rulebase, VEGA, or a
#  commercial system under ICH M7 - not through this notebook.

MUTAGENIC_NOTES = [
    (("nitro_group", "trinitro_aromatic", "gte_4_nitro",
      "trinitromethane"),
     "nitroaromatic: gut and liver nitroreductases reduce the nitro group "
     "to the nitroso and then the N-hydroxy compound, which forms covalent "
     "adducts with DNA bases. This is the single most common mutagenicity "
     "toxicophore and the reason nitroaromatics are usually avoided early.",
     "remove the nitro group, or replace it with a nitrile, sulfone or "
     "trifluoromethyl group if you need the electron withdrawal"),

    (("aniline", "aromatic_amine"),
     "aromatic amine: N-hydroxylated by CYP enzymes and then O-acetylated "
     "or O-sulfated, giving an ester that ionises to a nitrenium ion and "
     "alkylates guanine. The aromatic amine and the nitroaromatic converge "
     "on the same reactive intermediate from opposite directions.",
     "cap the nitrogen as an amide or sulfonamide, move it out of "
     "conjugation with the ring, or place ortho substituents that block "
     "N-hydroxylation"),

    (("N-nitroso", "nitrosamine"),
     "nitrosamine: alpha-hydroxylated by CYP2E1 to give a diazonium ion "
     "that methylates or alkylates DNA directly. This class is potent "
     "enough that regulators treat it as a cohort of concern with "
     "exposure limits far below ordinary impurity thresholds.",
     "do not draw an N-nitroso group; if a secondary amine could be "
     "nitrosated in formulation, that is a separate manufacturing concern"),

    (("Azo_group", "azo_aryl", "azo_amino", "azo_filter", "diazo_group",
      "activated_diazo", "diazonium", "diazo_carbonyl"),
     "azo compound: reduced by gut microbiota and hepatic azoreductases to "
     "two aromatic amines, so it carries the full aromatic-amine liability "
     "with it and delivers it to the colon.",
     "replace the azo link with an amide or a 1,2,3-triazole"),

    (("hydrazine", "acyl_hydrazine", "hydrazothiourea"),
     "hydrazine: oxidised to diazenes and carbon-centred radicals that "
     "alkylate DNA. Isoniazid carries this group and is still front-line "
     "tuberculosis therapy, which is the clearest illustration that an "
     "alert is not a verdict.",
     "use an amine, amide or N-heterocycle"),

    (("Polycyclic_aromatic_hydrocarbon", "polycyclic_aromatic"),
     "polycyclic aromatic: epoxidised by CYP1A1 and hydrolysed to the "
     "diol, then epoxidised again at the bay region; that diol epoxide is "
     "the ultimate carcinogen in this class, as with benzo[a]pyrene.",
     "break up the fused ring system, or introduce a heteroatom to change "
     "the oxidation pattern"),

    (("Azido_group", "azide"),
     "organic azide: energetic, and small azides are both mutagenic and a "
     "genuine detonation hazard on scale.",
     "use a triazole if you want the nitrogen, or an amine"),

    (("alkyl_halide", "alpha_halo_amine"),
     "alkyl halide: alkylates DNA directly by substitution, with no "
     "metabolic activation needed.",
     "remove the halide, or move it onto an aromatic ring where it is not "
     "a leaving group"),

    (("primary_halide_sulfate", "secondary_halide_sulfate",
      "tertiary_halide_sulfate", "sulfite_sulfate_ester"),
     "sulfonate or sulfate ester: an excellent leaving group on an sp3 "
     "carbon, so these alkylate DNA directly. Methyl and ethyl "
     "methanesulfonate are the standard positive controls in mutagenicity "
     "testing.",
     "replace the ester with an ether, or the leaving group with something "
     "that does not depart"),

    (("aziridine_diazirine", "oxaziridine"),
     "strained three-membered ring containing nitrogen: opened by DNA "
     "nucleophiles with no activation required.",
     "open the ring, or use azetidine or pyrrolidine"),
]


def mutagenic_note(desc):
    """(why, fix) if this catalogue entry is a mutagenicity toxicophore."""
    d = desc.lower()
    for keys, why, fix in MUTAGENIC_NOTES:
        if any(k.lower() in d for k in keys):
            return why, fix
    return None


# ----------------------------------------------------------------------
#  CONSIDERED AND NOT FLAGGED
# ----------------------------------------------------------------------
#  Groups students reasonably EXPECT to be alerts, which are deliberately
#  not. Silence looks like an oversight; saying "we looked at this and
#  here is why it is not flagged" is the teaching. These are NOT alerts:
#  they do not count towards the verdict, they do not colour atoms, and
#  they never turn a green molecule amber.

CONSIDERED = [
    dict(name="ketone",
         smarts="[#6;!$([CX3]=[OX1])][CX3](=[OX1])[#6;!$([CX3]=[OX1])]",
         skip_if=("ketoamide", "ketoester", "diketone", "trifluoromethyl",
                  "Michael", "alpha-halo"),
         note="a plain ketone is a weak electrophile. In water formaldehyde "
              "is more than 99.9% hydrated, acetaldehyde roughly half, "
              "acetone a few tenths of a percent - the same C=O across three "
              "orders of magnitude, because two alkyl groups both donate "
              "electron density and shield the carbonyl. Haloperidol, "
              "ketamine, ketoprofen and nabumetone all carry one. It becomes "
              "a real warhead only when the substituents make it one: "
              "conjugate it and you have a Michael acceptor, put a CF3 or a "
              "second carbonyl next to it and it acylates serine, put a "
              "halogen alpha to it and it alkylates. All four of those ARE "
              "flagged. The liability a plain ketone does carry is metabolic "
              "- reduction to the secondary alcohol by carbonyl reductase "
              "and the aldo-keto reductases - which this notebook does not "
              "cover."),

    dict(name="ester", smarts="[#6][CX3](=[OX1])[OX2][#6]",
         skip_if=("anhydride", "ketoester", "carbamic"),
         note="esters hydrolyse, but at rates spanning many orders of "
              "magnitude depending on what flanks them, and plasma esterases "
              "are often the point rather than the problem - that is what a "
              "prodrug is. Not flagged without more context."),

    dict(name="phenol", smarts="[OX2H][c]",
         skip_if=("quinone", "catechol"),
         note="a phenol is a metabolic soft spot - rapid glucuronidation and "
              "sulfation - rather than a stability or reactivity problem. "
              "Metabolism is outside this notebook's scope."),

    dict(name="aryl halide", smarts="[F,Cl,Br,I][c]",
         skip_if=("SNAr", "activated aryl"),
         note="a halogen on an electron-rich or neutral aromatic ring is "
              "stable and extremely common in drugs. It is only a liability "
              "when the ring is electron-poor enough for thiols to displace "
              "it, which is flagged separately as SNAr."),

    dict(name="carboxylic acid", smarts="[CX3](=[OX1])[OX2H]",
         skip_if=("carbamic",),
         note="ionised at physiological pH, which costs permeability and "
              "usually rules out the CNS, but it is not a stability or "
              "reactivity problem. A design consideration, not an alert."),

    dict(name="basic amine",
         smarts="[NX3;H0,H1;!$(N[CX3]=[OX1]);!$(N[SX4]);!$(N=*);!$(N[OX1])]",
         skip_if=("mustard", "aminal", "hydroxylamine", "hydrazine",
                  "nitroso"),
         note="a basic amine is protonated at physiological pH. That drives "
              "solubility and target engagement for a very large fraction of "
              "marketed drugs; it also carries hERG and lysosomal trapping "
              "risk at high basicity and lipophilicity. Judgement, not an "
              "alert."),

    dict(name="nitrile", smarts="[NX1]#[CX2][#6]",
         skip_if=("acyl cyanide",),
         note="aryl and alkyl nitriles are stable and are a standard "
              "replacement FOR an aldehyde. Only an activated nitrile next "
              "to a carbonyl is electrophilic, and that is flagged "
              "separately."),
]


def considered_and_cleared(mol, alerts):
    """Groups present that were looked at and deliberately not flagged."""
    out = []
    fired = " ".join(h["name"] for h in alerts).lower()
    for c in CONSIDERED:
        if any(s.lower() in fired for s in c.get("skip_if", ())):
            continue
        patt = Chem.MolFromSmarts(c["smarts"])
        if patt is None or not mol.HasSubstructMatch(patt):
            continue
        n = len(mol.GetSubstructMatches(patt))
        out.append(dict(name=c["name"], n=n, note=c["note"]))
    return out
# ======================================================================
#  STAGE 1 - STRUCTURAL VALIDITY
# ======================================================================
#  This runs BEFORE any chemistry. A molecule that fails here is not a
#  molecule at all, and nothing downstream can run on it.

_VALENCE_RE = re.compile(r"atom # (\d+) (\w+), (\d+), is greater than permitted")
_MAX_VAL = {"C": 4, "N": 3, "O": 2, "S": 6, "P": 5, "F": 1,
            "Cl": 1, "Br": 1, "I": 1, "B": 3, "H": 1}


def _describe_atom(mol, idx):
    """'C3 (2 H, in a ring)' - readable, 1-based to match the drawing."""
    try:
        a = mol.GetAtomWithIdx(idx)
    except Exception:
        return f"atom {idx + 1}"
    sym = a.GetSymbol()
    nH = a.GetTotalNumHs()
    bits = [f"{nH} H"] if nH else []
    if a.IsInRing():
        bits.append("in a ring")
    return f"{sym}{idx + 1}" + (f" ({', '.join(bits)})" if bits else "")


_ELEMENT = {"C": "carbon", "N": "nitrogen", "O": "oxygen", "S": "sulfur",
            "P": "phosphorus", "B": "boron"}


def _el(sym):
    return _ELEMENT.get(sym, "atom")


def _bond_breakdown(a):
    """(total_valence, 'one single and one double bond') for one atom."""
    singles = doubles = triples = aromatic = 0
    for b in a.GetBonds():
        d = b.GetBondTypeAsDouble()
        if d == 1.0:   singles += 1
        elif d == 2.0: doubles += 1
        elif d == 3.0: triples += 1
        else:          aromatic += 1
    order = sum(b.GetBondTypeAsDouble() for b in a.GetBonds())
    words = []
    def _n(k):
        return {1: "one", 2: "two", 3: "three", 4: "four", 5: "five"}.get(k, str(k))
    def _w(k, kind):
        return f"{_n(k)} {kind} bond" + ("s" if k > 1 else "")
    if singles:  words.append(_w(singles, "single"))
    if doubles:  words.append(_w(doubles, "double"))
    if triples:  words.append(_w(triples, "triple"))
    if aromatic: words.append(_w(aromatic, "aromatic"))
    if not words:
        txt = "no bonds to other atoms"
    elif len(words) == 1:
        txt = words[0]
    else:
        txt = ", ".join(words[:-1]) + " and " + words[-1]
    return order + a.GetTotalNumHs(), order, txt


def check_validity(smi):
    """
    Returns a dict:
      ok        True if RDKit can build and sanitise the molecule
      stage     'parse' / 'chemistry' / 'ok'
      problems  list of dicts: kind, atoms (0-based), plain, fix
      mol       a drawable molecule, or None if even the graph failed
    """
    out = dict(ok=False, stage="parse", problems=[], mol=None, smiles=smi)

    if not smi or not smi.strip():
        out["problems"] = [dict(kind="EMPTY", atoms=[],
                                plain="You did not give a SMILES string.",
                                fix="Paste the SMILES your drawing program "
                                    "exported.")]
        return out

    smi = smi.strip()
    out["smiles"] = smi

    # --- 1a. can the graph be built at all? (syntax) -------------------
    mol = Chem.MolFromSmiles(smi, sanitize=False)
    if mol is None:
        out["problems"] = [dict(
            kind="SYNTAX", atoms=[],
            plain="RDKit cannot read this as SMILES at all. The text itself is "
                  "malformed - usually an unclosed ring number, an unbalanced "
                  "bracket, or a stray character.",
            fix="Check that every ring-opening digit has a partner and every "
                "( has a ). Re-export from your drawing program rather than "
                "typing it by hand.")]
        return out

    # --- 1b. does the chemistry make sense? ---------------------------
    mol.UpdatePropertyCache(strict=False)
    probs = Chem.DetectChemistryProblems(mol)
    out["mol"] = mol

    if not probs:
        clean = Chem.MolFromSmiles(smi)
        if clean is None:                       # belt and braces
            out["stage"] = "chemistry"
            out["problems"] = [dict(kind="OTHER", atoms=[],
                                    plain="RDKit rejected this molecule but "
                                          "gave no specific reason.",
                                    fix="Simplify the structure and add pieces "
                                        "back until it breaks.")]
            return out
        out.update(ok=True, stage="ok", mol=clean)
        return out

    out["stage"] = "chemistry"
    for p in probs:
        kind = p.GetType()
        msg = p.Message()

        if kind == "AtomValenceException":
            idx = p.GetAtomIdx()
            mm = _VALENCE_RE.search(msg)
            try:
                a = mol.GetAtomWithIdx(idx)
            except Exception:
                a = None
            sym = mm.group(2) if mm else (a.GetSymbol() if a else "the atom")
            found = int(mm.group(3)) if mm else None
            allowed = _MAX_VAL.get(sym)

            if a is not None:
                total, order, bond_txt = _bond_breakdown(a)
                nH = a.GetTotalNumHs()
                found = found if found is not None else int(total)
                h_txt = (f"carries {nH} hydrogen{'s' if nH != 1 else ''} and "
                         if nH else "has ")
                plain = (f"{sym}{idx + 1} {h_txt}{bond_txt} to other atoms, "
                         f"which is {found} in total. "
                         + (f"{sym} allows {allowed}."
                            if allowed else f"That is more than {sym} allows."))
            else:
                plain = f"{_describe_atom(mol, idx)} has too many bonds."

            # a bracketed atom pins its hydrogen count, which is the usual
            # cause: [CH2] says EXACTLY two H, and the ring bonds are extra
            if a is not None and a.GetNoImplicit() and a.GetNumExplicitHs():
                nH = a.GetNumExplicitHs()
                def _br(k):                       # [C], [CH], [CH2] ...
                    return f"[{sym}]" if k <= 0 else \
                           f"[{sym}H]" if k == 1 else f"[{sym}H{k}]"
                over = found - (allowed or 4)
                drop = max(nH - over, 0)
                has_multiple = any(b.GetBondTypeAsDouble() > 1.0
                                   for b in a.GetBonds())
                # Lead with the CHEMISTRY. The bracket is only how the
                # mistake got expressed; the decision the student has to
                # make is about hydrogens and bond order. Do NOT gloss
                # "CH2" as "2 hydrogens" - the formula already says that,
                # and restating it makes the sentence read as though two
                # separate things were drawn.
                maxv = allowed or 4
                left = maxv - nH
                w = {0: "no", 1: "one", 2: "two", 3: "three",
                     4: "four", 5: "five", 6: "six"}
                fix = f"You have drawn this {_el(sym)} as {sym}H{nH} "
                if has_multiple:
                    fix += (f"and also given it a double bond. {sym} has "
                            f"{w.get(maxv, maxv)} bonds in all, so "
                            f"{w.get(nH, nH)} hydrogen"
                            f"{'s' if nH != 1 else ''} "
                            f"leave{'s' if nH == 1 else ''} "
                            + ("none at all for its neighbours"
                               if left <= 0 else
                               f"only {w.get(left, left)} for its neighbours")
                            + " - and a double bond uses two on its own. ")
                else:
                    nb = a.GetDegree()
                    fix += (f"and bonded it to {w.get(nb, nb)} "
                            f"neighbour{'s' if nb != 1 else ''}, which is "
                            f"{w.get(over, over)} too many. ")
                fix += (f"Decide which you meant: write {_br(drop)}, or drop "
                        f"the square brackets and write plain {sym} so the "
                        f"software works the hydrogens out for you")
                if has_multiple:
                    fix += ", or remove the double bond"
                fix += (f". The brackets are what pinned it - in SMILES "
                        f"{_br(nH)} means exactly {w.get(nH, nH)} "
                        f"hydrogen{'s' if nH != 1 else ''}.")
                if a.IsInRing():
                    fix += (" If you were drawing benzene, the whole ring is "
                            "just c1ccccc1, with no brackets anywhere.")
            else:
                fix = ("Delete one bond at that atom, or one of its explicit "
                       "hydrogens. If you meant a charged atom, write the "
                       "charge in brackets, for example [NH4+].")
            out["problems"].append(dict(
                kind="VALENCE", atoms=[idx], highlight=[idx], plain=plain,
                fix=fix))

        elif kind == "KekulizeException":
            idxs = list(p.GetAtomIndices())

            # RDKit lists the atoms it could not kekulise - but the atom that
            # actually broke the ring is often NOT in that list, because an
            # sp3 carbon is not part of the aromatic system at all. So widen
            # the search to every ring that touches a flagged atom.
            try:
                Chem.FastFindRings(mol)
            except Exception:
                pass
            flagged = set(idxs)
            ring_members = set()
            try:
                for ring in mol.GetRingInfo().AtomRings():
                    if flagged & set(ring):
                        ring_members |= set(ring)
            except Exception:
                pass
            ring_members |= flagged

            sat, nonarom = [], []
            for i in sorted(ring_members):
                try:
                    a = mol.GetAtomWithIdx(i)
                except Exception:
                    continue
                # an aromatic ring carbon may carry at most one H
                if a.GetSymbol() == "C" and a.GetTotalNumHs() >= 2:
                    sat.append(i)
                elif not a.GetIsAromatic() and any(
                        nb.GetIsAromatic() for nb in a.GetNeighbors()):
                    nonarom.append(i)

            culprits = sat or nonarom
            if sat:
                names = ", ".join(_describe_atom(mol, i) for i in sat)
                plain = (f"You drew this ring as aromatic, but {names} is "
                         f"saturated - it carries two hydrogens, so it is a "
                         f"CH2 and cannot be part of an aromatic ring. An "
                         f"aromatic ring carbon can carry at most one H.")
                fix = ("Decide which ring you meant. For benzene, remove one "
                       "hydrogen from that carbon so every ring carbon is CH: "
                       "c1ccccc1. For the saturated ring, draw the whole ring "
                       "with capital letters and single bonds: C1CCCCC1. You "
                       "cannot mix the two in one ring.")
            elif nonarom:
                names = ", ".join(_describe_atom(mol, i) for i in nonarom)
                plain = (f"This ring is drawn as aromatic, but {names} is "
                         f"written as an aliphatic atom (a capital letter in "
                         f"SMILES). Every atom in an aromatic ring has to be "
                         f"part of the same pi system.")
                fix = ("Either make that atom aromatic too (lower case in "
                       "SMILES), or draw the whole ring as saturated with "
                       "capital letters and single bonds.")
            else:
                names = ", ".join(_describe_atom(mol, i) for i in idxs)
                plain = (f"This ring is drawn as aromatic but no alternating "
                         f"pattern of single and double bonds can be written "
                         f"for it. The atoms involved are {names}.")
                fix = ("Count the pi electrons: an aromatic ring needs 4n+2. "
                       "Either add the missing double bond, draw an NH where "
                       "a pyrrole-type nitrogen belongs, or write the ring "
                       "as saturated with capital letters.")
            # `atoms` is what RDKit flagged; `highlight` is what the student
            # should look at, which is often not the same thing.
            out["problems"].append(dict(kind="KEKULIZE", atoms=idxs,
                                        highlight=culprits or idxs,
                                        plain=plain, fix=fix))
        else:
            out["problems"].append(dict(
                kind=kind, atoms=[], plain=msg,
                fix="Rebuild this part of the molecule in your drawing "
                    "program."))
    return out


# ======================================================================
#  STAGE 2 - ALERTS  (only reached once the molecule is valid)
# ======================================================================

_PAINS_CAT = None
_BRENK_CAT = None
_NIH_CAT = None

def _catalogs():
    global _PAINS_CAT, _BRENK_CAT, _NIH_CAT
    if _PAINS_CAT is None:
        p = FilterCatalogParams(); p.AddCatalog(FilterCatalogParams.FilterCatalogs.PAINS)
        _PAINS_CAT = FilterCatalog.FilterCatalog(p)
        b = FilterCatalogParams(); b.AddCatalog(FilterCatalogParams.FilterCatalogs.BRENK)
        _BRENK_CAT = FilterCatalog.FilterCatalog(b)
        n = FilterCatalogParams(); n.AddCatalog(FilterCatalogParams.FilterCatalogs.NIH)
        _NIH_CAT = FilterCatalog.FilterCatalog(n)
    return _PAINS_CAT, _BRENK_CAT, _NIH_CAT


def _ring_strain(mol):
    """Anti-Bredt and small-ring flags that SMARTS handles badly."""
    hits = []
    ri = mol.GetRingInfo()

    # bridgehead sp2 carbon in a small bridged system (anti-Bredt)
    patt = Chem.MolFromSmarts("[x3,x4]")       # atom with 3 or 4 ring bonds
    for match in mol.GetSubstructMatches(patt):
        i = match[0]
        a = mol.GetAtomWithIdx(i)
        if a.GetSymbol() != "C":
            continue
        if not any(b.GetBondTypeAsDouble() == 2.0 for b in a.GetBonds()):
            continue
        sizes = [len(r) for r in ri.AtomRings() if i in r]
        if sizes and max(sizes) <= 7:
            hits.append(dict(
                name="anti-Bredt alkene", cat="STRAIN", sev="HIGH",
                atoms=[i],
                why=f"{_describe_atom(mol, i)} is a bridgehead carrying a "
                    f"double bond in a bridged ring system of {max(sizes)} "
                    f"atoms; the p orbitals cannot align, so this alkene "
                    f"cannot exist (Bredt's rule)",
                fix="move the double bond off the bridgehead, or enlarge the "
                    "ring to eight atoms or more", iso=""))
            break

    # fused three-membered ring on an aromatic
    patt2 = Chem.MolFromSmarts("[cR2]1[cR2]C1")
    if mol.HasSubstructMatch(patt2):
        hits.append(dict(
            name="cyclopropa-fused arene", cat="STRAIN", sev="MEDIUM",
            atoms=list(mol.GetSubstructMatch(patt2)),
            why="a three-membered ring fused directly onto an aromatic ring is "
                "severely strained and opens readily",
            fix="use a four- or five-membered fused ring", iso=""))
    return hits


# ======================================================================
#  STAGE 1.5 - IS THE DRAWING COMPLETE?
# ======================================================================
#  A structure can parse, sanitise, and trip no alert, and still not
#  describe a single compound. A 1,4-disubstituted cyclohexane drawn with
#  no stereochemistry is two different molecules with two different
#  shapes. This is not a chemistry error and not a stability problem, so
#  it is neither Stage 1 nor Stage 2 - it is its own question.

def check_stereo(mol):
    """Unspecified stereocentres and double-bond geometry."""
    out = []
    try:
        info = Chem.FindPotentialStereo(mol)
    except Exception:
        return out
    atoms, bonds = [], []
    for e in info:
        specified = str(e.specified)
        if "Unspecified" not in specified:
            continue
        t = str(e.type)
        if "Atom" in t:
            atoms.append(e.centeredOn)
        elif "Bond" in t:
            bonds.append(e.centeredOn)

    if atoms:
        names = ", ".join(_describe_atom(mol, i) for i in atoms[:8])
        n = len(atoms)
        out.append(dict(
            name=f"{n} unspecified stereocentre{'s' if n > 1 else ''}",
            cat="INCOMPLETE", sev="NOTE", atoms=atoms,
            why=f"the configuration at {names} is not defined, so this "
                f"drawing describes {2 ** min(n, 10)} different "
                f"stereoisomer{'s' if n > 1 else ''} rather than one "
                f"compound. They have different shapes, they dock "
                f"differently, and they are separable. In a ring - a "
                f"1,4-disubstituted cyclohexane, say - this is the cis "
                f"versus trans question.",
            fix="decide which isomer you mean and draw the wedge and dash "
                "bonds, or state explicitly that you intend the mixture",
            iso=""))
    if bonds:
        n = len(bonds)
        out.append(dict(
            name=f"{n} unspecified double bond{'s' if n > 1 else ''}",
            cat="INCOMPLETE", sev="NOTE", atoms=[],
            why="the geometry of a double bond is not defined, so E and Z "
                "are both being described at once; they are different "
                "compounds with different shapes",
            fix="draw the double bond as cis or trans", iso=""))
    return out


def scan_alerts(mol):
    """All stage-2 hits for a sanitised molecule, worst first."""
    hits = []

    for a, patt in _COMPILED:
        matches = mol.GetSubstructMatches(patt)
        if matches:
            # colour only the atoms that ARE the alert, not the context
            # atoms the SMARTS needed in order to match
            core = a.get("core")
            if core:
                atoms = sorted({m[i] for m in matches for i in core
                                if i < len(m)})
            else:
                atoms = sorted({i for m in matches for i in m})
            hits.append(dict(name=a["name"], cat=a["cat"], sev=a["sev"],
                             atoms=atoms, why=a["why"], fix=a["fix"],
                             iso=a["iso"], n=len(matches)))

    pains, brenk, nih = _catalogs()

    for entry in pains.GetMatches(mol):
        desc = entry.GetDescription()
        atoms = []
        for fm in entry.GetFilterMatches(mol):
            atoms += [int(b) for _, b in fm.atomPairs]
        mut = mutagenic_note(desc)
        hits.append(dict(
            catalog=True, source="PAINS",
            name=f"PAINS (pan-assay interference): {desc}",
            cat="MUTAGENIC" if mut else "ASSAY", sev="MEDIUM",
            atoms=sorted(set(atoms)),
            why=mut[0] if mut else
                "a pan-assay interference compound: this substructure appears "
                "again and again as a hit across unrelated assays. It usually "
                "works by forming colloidal aggregates, by redox chemistry, or "
                "by interfering with the readout itself, rather than by "
                "binding the target. Roughly 5% of approved drugs contain one.",
            fix=mut[1] if mut else
                "confirm the activity in an orthogonal assay, and with a "
                "detergent control, before believing it",
            iso="", n=1))

    seen = {h["name"] for h in hits}
    for cat_obj, label, sev in ((brenk, "Brenk", "MEDIUM"), (nih, "NIH", "NOTE")):
        for entry in cat_obj.GetMatches(mol):
            desc = entry.GetDescription()
            nm = f"{label}: {desc}"
            if nm in seen:
                continue
            seen.add(nm)
            atoms = []
            for fm in entry.GetFilterMatches(mol):
                atoms += [int(b) for _, b in fm.atomPairs]
            mut = mutagenic_note(desc)
            hits.append(dict(
                catalog=True, source=label,
                name=nm, cat="MUTAGENIC" if mut else "ASSAY",
                sev="MEDIUM" if mut else sev, atoms=sorted(set(atoms)),
                why=mut[0] if mut else
                    ("on the Brenk list of substructures excluded from "
                     "screening libraries for reactivity, instability or "
                     "toxicity" if label == "Brenk" else
                     "on the NIH list of substructures that have given "
                     "trouble in practice"),
                fix=mut[1] if mut else "", iso="", n=1))

    hits += _ring_strain(mol)
    hits = _merge_duplicates(hits)
    hits.sort(key=lambda h: (SEV_ORDER.get(h["sev"], 9), h["name"]))
    return hits


def _norm(name):
    """'Brenk: Michael_acceptor_1' -> 'michael acceptor'"""
    n = re.sub(r"^(PAINS|PAINS \(pan-assay interference\)|Brenk|NIH):\s*",
               "", name, flags=re.I)
    n = re.sub(r"\(\d+\)$", "", n)          # PAINS entries end in (92)
    n = n.replace("_", " ").strip().lower()
    n = re.sub(r"\s+\d+$", "", n)            # Michael acceptor 1 -> ...
    n = re.sub(r"\s+group$", "", n)
    return n.strip()


def _merge_duplicates(hits):
    """
    One aldehyde should be ONE row, not three. When a catalogue entry says
    the same thing as a curated pattern, fold it in as corroboration
    instead of repeating it. This also keeps the colouring honest: the
    curated pattern knows which atoms ARE the alert, while a catalogue
    SMARTS often drags in a neighbouring atom that is only context.
    """
    curated = [h for h in hits if not h.get("catalog")]
    catalog = [h for h in hits if h.get("catalog")]
    by_norm = {}
    for h in curated:
        by_norm.setdefault(_norm(h["name"]), h)

    out = list(curated)
    for h in catalog:
        key = _norm(h["name"])
        owner = by_norm.get(key)
        if owner is None:                      # try loose containment
            for k, cand in by_norm.items():
                if k and (k in key or key in k):
                    owner = cand
                    break
        if owner is not None:
            owner.setdefault("also", []).append(h["source"])
        else:
            out.append(h)
    return out
# ======================================================================
#  DRAWING
# ======================================================================

SEV_RGB = {"BLOCKER": (0.55, 0.00, 0.00), "HIGH": (0.90, 0.26, 0.21),
           "MEDIUM": (0.95, 0.70, 0.25), "NOTE": (0.72, 0.75, 0.73)}

# A pale "highlighter pen" wash, used behind an atom LABEL. A solid
# circle behind text clips it; a pale one sits underneath and stays
# readable, which is what lets the label live ON the atom at all.
PALE_MARK = (1.00, 0.80, 0.78)


def draw_mol(mol, highlight=(), size=(470, 380), legend="", kekulize=True,
             number_atoms=False, atom_colours=None, labels=None, notes=None,
             label_colours=None, radius=0.32):
    """
    SVG of the molecule.
      highlight     atoms to circle
      atom_colours  {atom_idx: (r,g,b)} - overrides the default red
      notes         {atom_idx: "3 = CH2"} - written BESIDE the atom. Use
                    this, not `labels`, to say what an atom is: an
                    atomLabel sits ON the atom and RDKit grows the
                    highlight to fit it, which turns a neat circle into a
                    blob that swallows the bonds. A note keeps the circle
                    small and still tells the student what they are
                    looking at.
      labels         {atom_idx: "CH2"} - replaces the atom symbol itself.
                    Inflates the highlight; avoid for flagged atoms.
    """
    if mol is None:
        return "<p style='color:#8b0000'>nothing to draw</p>"
    m = Chem.Mol(mol)

    for i, txt in (labels or {}).items():
        if i < m.GetNumAtoms():
            m.GetAtomWithIdx(int(i)).SetProp("atomLabel", txt)
    if number_atoms:
        # 1-based numbers so the picture matches the report text
        for a in m.GetAtoms():
            a.SetProp("atomNote", str(a.GetIdx() + 1))
    for i, txt in (notes or {}).items():          # overrides the bare number
        if i < m.GetNumAtoms():
            m.GetAtomWithIdx(int(i)).SetProp("atomNote", txt)

    try:
        AllChem.Compute2DCoords(m)
    except Exception:
        pass
    d = rdMolDraw2D.MolDraw2DSVG(size[0], size[1])
    o = d.drawOptions()
    o.addAtomIndices = False
    o.highlightRadius = radius        # keep the marker tight around the atom
    o.annotationFontScale = 0.75
    o.highlightBondWidthMultiplier = 16

    hl = [int(i) for i in highlight if i < m.GetNumAtoms()]
    cols = {}
    for i in hl:
        cols[i] = (atom_colours or {}).get(i, SEV_RGB["HIGH"])
    try:
        rdMolDraw2D.PrepareAndDrawMolecule(
            d, m, highlightAtoms=hl, highlightAtomColors=cols,
            legend=legend, kekulize=kekulize)
    except Exception:
        try:                                   # broken molecule: skip kekulize
            rdMolDraw2D.PrepareAndDrawMolecule(
                d, m, highlightAtoms=hl, highlightAtomColors=cols,
                legend=legend, kekulize=False)
        except Exception:
            d.DrawMolecule(m)
    d.FinishDrawing()
    svg = d.GetDrawingText()
    # RDKit tags every glyph of atom i with class='atom-i', so the label
    # text can be recoloured after the fact. Wrapped in try/except: if a
    # future RDKit changes the markup we simply keep the default colour
    # rather than failing to draw.
    if label_colours:
        try:
            for i, col in label_colours.items():
                svg = re.sub(
                    r"(<path class='atom-%d'[^>]*?fill=')#[0-9A-Fa-f]{6}(')"
                    % int(i), r"\g<1>" + col + r"\g<2>", svg)
        except Exception:
            pass
    return svg


def problem_notes(mol, problems):
    """
    What to write BESIDE each flagged atom in a Stage 1 failure.
    A red circle on a bare vertex tells a student WHERE but not WHAT, so
    the note carries both: "3 = CH2". Keep it short - it is an annotation,
    not an explanation. The arithmetic belongs in the text panel.
    """
    notes = {}
    if mol is None:
        return notes
    for p in problems:
        for i in p.get("highlight", p.get("atoms", [])):
            try:
                a = mol.GetAtomWithIdx(int(i))
            except Exception:
                continue
            sym, nH = a.GetSymbol(), a.GetTotalNumHs()
            # NO separator and NO number here. "10 = CH2" reads as a double
            # bond to an exocyclic methylene - an "=" next to a structure is
            # a bond, not a label. The number goes in the legend instead.
            notes[i] = f"{sym}H{nH}" if nH > 1 else (f"{sym}H" if nH == 1
                                                     else sym)
    return notes


def problem_legend(mol, problems):
    """'red = the atom to look at (C10 in the report)'"""
    idxs = sorted({i for p in problems
                   for i in p.get("highlight", p.get("atoms", []))})
    if mol is None or not idxs:
        return "red = the atom to look at"
    names = ", ".join(f"{mol.GetAtomWithIdx(int(i)).GetSymbol()}{i + 1}"
                      for i in idxs[:4])
    return f"the marked atom is the problem  ({names} in the report)"


def _panel(title, body, colour="#2c5c40"):
    return (f"<div style='border-left:5px solid {colour};background:#f7f9f7;"
            f"padding:10px 16px;margin:10px 0;border-radius:4px'>"
            f"<b style='color:{colour}'>{title}</b><br>{body}</div>")


# ======================================================================
#  VERDICT BANNER
# ======================================================================
#  Four states, not three. "Not a molecule" is categorically different
#  from "a molecule with problems" and must not share a colour with it.
VERDICT = {
    # dark red, not black: "not a molecule" is the MOST severe outcome,
    # and black reads as muted or informational. The darker shade keeps it
    # distinguishable from the HIGH banner without demoting it.
    "INVALID": ("#8b0000", "NOT A VALID MOLECULE",
                "This did not parse or did not sanitise. Nothing downstream "
                "can run on it."),
    "HIGH":    ("#c0392b", "HIGH-SEVERITY ALERT",
                "Something here will not survive as drawn, or is a hard "
                "covalent liability. Fix it, or be ready to say why you meant "
                "it."),
    "MEDIUM":  ("#d68910", "WORTH JUSTIFYING",
                "No high-severity problem, but something here is worth "
                "defending. You should be able to say why it is there."),
    "CLEAN":   ("#2c5c40", "NO ALERTS MATCHED",
                "Nothing matched a pattern this notebook knows about. That is "
                "the absence of a flag, not evidence of a good molecule - it "
                "says nothing about potency, selectivity, metabolism or "
                "whether anyone can make it."),
}


def _banner(state, extra=""):
    colour, title, blurb = VERDICT[state]
    return (f"<div style='background:{colour};color:#ffffff;padding:14px 22px;"
            f"border-radius:8px;margin:14px 0;font-family:system-ui,sans-serif'>"
            f"<div style='font-size:22px;font-weight:bold'>{title}</div>"
            f"<div style='font-size:14px;opacity:0.93;margin-top:4px'>{blurb}"
            f"{extra}</div></div>")


# ======================================================================
#  THE MAIN ENTRY POINT
# ======================================================================

def check(smi, show=True, name=""):
    """
    Run every stage on one molecule.
    Returns: valid, problems, stereo, alerts, verdict, table, mol
    """
    res = dict(name=name or "your molecule", smiles=smi, valid=False,
               problems=[], stereo=[], alerts=[], considered=[],
               verdict="INVALID", table=None, mol=None)

    v = check_validity(smi)
    res["problems"] = v["problems"]
    res["mol"] = v["mol"]
    res["valid"] = v["ok"]

    # ---------------- stage 1 failed --------------------------------
    if not v["ok"]:
        res["verdict"] = "INVALID"
        if show:
            bad = sorted({i for p in v["problems"]
                          for i in p.get("highlight", p["atoms"])})
            display(HTML(_banner("INVALID")))
            html = [f"<div style='font-family:system-ui,sans-serif'>"
                    f"<code style='font-size:13px'>{v['smiles']}</code>"]
            for p in v["problems"]:
                html.append(_panel(
                    f"{p['kind']} &nbsp;&mdash;&nbsp; what is wrong",
                    f"{p['plain']}<br><br><b>How to fix it:</b> {p['fix']}",
                    "#8b0000"))
            html.append("<p style='color:#555;font-size:13px'>Stage 2 did not "
                        "run. Stability and reactivity alerts can only be "
                        "checked once the molecule itself is valid.</p></div>")
            display(HTML("".join(html)))
            if v["mol"] is not None:
                display(SVG(draw_mol(
                    v["mol"], highlight=bad,
                    # The formula goes ON the atom as a label, over a PALE
                    # wash rather than a solid circle - a solid fill clips
                    # the text sitting on top of it. The label itself is
                    # recoloured red, so the marker reads without the
                    # label having to fight it.
                    labels=problem_notes(v["mol"], v["problems"]),
                    atom_colours={i: PALE_MARK for i in bad},
                    label_colours={i: "#C0392B" for i in bad},
                    radius=0.40,
                    legend=problem_legend(v["mol"], v["problems"]),
                    kekulize=False, number_atoms=False)))
        return res

    mol = v["mol"]
    res["mol"] = mol
    res["stereo"] = check_stereo(mol)
    alerts = scan_alerts(mol)
    res["alerts"] = alerts
    res["considered"] = considered_and_cleared(mol, alerts)

    if any(h["sev"] == "HIGH" for h in alerts):
        res["verdict"] = "HIGH"
    elif any(h["sev"] == "MEDIUM" for h in alerts):
        res["verdict"] = "MEDIUM"
    else:
        res["verdict"] = "CLEAN"

    rows = []
    for h in alerts + res["stereo"]:
        rows.append(dict(
            severity=h["sev"], category=h["cat"], alert=h["name"],
            atoms=" ".join(str(i + 1) for i in h["atoms"][:12]) or "-",
            why=h["why"], suggested_fix=h["fix"] or "-",
            bioisostere_lookup=h.get("iso", "") or "-"))
    res["table"] = pd.DataFrame(rows)

    if show:
        _render(res)
    return res


def _render(res):
    mol, alerts, stereo = res["mol"], res["alerts"], res["stereo"]
    n_high = sum(1 for h in alerts if h["sev"] == "HIGH")
    n_med = sum(1 for h in alerts if h["sev"] == "MEDIUM")
    n_note = sum(1 for h in alerts if h["sev"] == "NOTE")

    extra = ""
    if alerts:
        extra = (f"<br><b>{n_high}</b> high &nbsp;&middot;&nbsp; "
                 f"<b>{n_med}</b> medium &nbsp;&middot;&nbsp; "
                 f"<b>{n_note}</b> note")
    display(HTML(_banner(res["verdict"], extra)))

    display(HTML(
        f"<div style='font-family:system-ui,sans-serif'>"
        f"<code style='font-size:13px'>{Chem.MolToSmiles(mol)}</code>"
        f"<p style='font-size:13px;color:#555'>"
        f"MW {Descriptors.MolWt(mol):.1f} &nbsp;&middot;&nbsp; "
        f"cLogP {Descriptors.MolLogP(mol):.2f} &nbsp;&middot;&nbsp; "
        f"TPSA {Descriptors.TPSA(mol):.1f} &nbsp;&middot;&nbsp; "
        f"heavy atoms {mol.GetNumHeavyAtoms()}</p></div>"))

    # colour every flagged atom by the WORST severity that hit it, so one
    # picture can show a red aldehyde and an amber PAINS ring at once
    worst = {}
    for h in alerts:
        for i in h["atoms"]:
            if SEV_ORDER.get(h["sev"], 9) < SEV_ORDER.get(worst.get(i, "NOTE"), 9) \
               or i not in worst:
                worst[i] = h["sev"]
    colours = {i: SEV_RGB.get(s, SEV_RGB["NOTE"]) for i, s in worst.items()}
    # name only the colours actually on the picture - a legend that lists
    # amber and grey when neither appears invites the reader to hunt for
    # something that is not there
    present = {s for s in worst.values()}
    key = [("HIGH", "red = high"), ("MEDIUM", "amber = medium"),
           ("NOTE", "grey = note")]
    legend = "   ".join(t for s, t in key if s in present) or "no alerts matched"
    display(SVG(draw_mol(mol, highlight=sorted(worst), atom_colours=colours,
                         legend=legend)))

    blocks = []
    for h in alerts:
        c = SEV_COLOUR.get(h["sev"], "#2c5c40")
        pos = " ".join(str(i + 1) for i in h["atoms"][:12]) or "-"
        also = h.get("also") or []
        also_txt = (f" &nbsp;&middot;&nbsp; also flagged by "
                    f"{', '.join(sorted(set(also)))}" if also else "")
        body = (f"<span style='color:#555;font-size:12px'>{h['cat']} "
                f"&nbsp;&middot;&nbsp; atoms {pos}{also_txt}</span><br>"
                f"{h['why']}")
        if h["fix"]:
            body += f"<br><br><b>Suggested fix:</b> {h['fix']}"
        if h.get("iso"):
            body += (f"<br><b>Bioisostere Explorer:</b> look up "
                     f"<code>{h['iso']}</code> for replacements.")
        blocks.append(_panel(f"[{h['sev']}] {h['name']}", body, c))

    # stereo is its own question and gets its own colour
    for h in stereo:
        blocks.append(_panel(
            f"[INCOMPLETE] {h['name']}",
            f"<span style='color:#555;font-size:12px'>this is not an alert - "
            f"the molecule is valid and may be perfectly stable</span><br>"
            f"{h['why']}<br><br><b>What to do:</b> {h['fix']}", "#2e6ea6"))

    if blocks:
        display(HTML("<div style='font-family:system-ui,sans-serif'>"
                     + "".join(blocks) + "</div>"))

    # groups that are present, were looked at, and are deliberately NOT
    # alerts. Silence looks like an oversight; this says otherwise.
    seen = res.get("considered") or []
    if seen:
        parts = []
        for c in seen:
            times = f" &times;{c['n']}" if c["n"] > 1 else ""
            parts.append(f"<li style='margin-bottom:7px'><b>{c['name']}</b>"
                         f"{times} &mdash; {c['note']}</li>")
        rows = "".join(parts)
        display(HTML(
            "<div style='font-family:system-ui,sans-serif;border-left:5px "
            "solid #8a8f8c;background:#f6f7f6;padding:10px 18px;margin:10px 0;"
            "border-radius:4px;font-size:14px'>"
            "<b style='color:#5a605c'>Considered and not flagged</b>"
            "<div style='color:#666;font-size:12.5px;margin:3px 0 7px 0'>"
            "These groups are in your molecule. They are not alerts, they do "
            "not affect the verdict, and they are listed so you know they "
            "were looked at rather than missed.</div>"
            f"<ul style='margin:0;padding-left:20px'>{rows}</ul></div>"))
# ======================================================================
#  BATCH MODE  - the research half
# ======================================================================

def check_batch(smiles_list, names=None, show=True, top=15):
    """
    Run both stages over many molecules. Built for generative-model output:
    it reports the validity rate AND which failure modes dominate, not just
    a single percentage.
    Returns (per_molecule DataFrame, summary dict).
    """
    if isinstance(smiles_list, str) and os.path.exists(smiles_list):
        with open(smiles_list) as fh:
            smiles_list = [ln.split()[0] for ln in fh
                           if ln.strip() and not ln.startswith("#")]
    names = names or [f"mol_{i+1}" for i in range(len(smiles_list))]

    rows, fail_kinds, alert_counts = [], {}, {}
    for nm, smi in zip(names, smiles_list):
        v = check_validity(smi)
        if not v["ok"]:
            kinds = sorted({p["kind"] for p in v["problems"]})
            for k in kinds:
                fail_kinds[k] = fail_kinds.get(k, 0) + 1
            rows.append(dict(name=nm, smiles=smi, valid=False,
                             failure=";".join(kinds), n_high=0, n_medium=0,
                             n_note=0, alerts=""))
            continue
        hits = scan_alerts(v["mol"])
        st = check_stereo(v["mol"])
        for h in hits:
            alert_counts[h["name"]] = alert_counts.get(h["name"], 0) + 1
        rows.append(dict(
            name=nm, smiles=Chem.MolToSmiles(v["mol"]), valid=True, failure="",
            n_high=sum(1 for h in hits if h["sev"] == "HIGH"),
            n_medium=sum(1 for h in hits if h["sev"] == "MEDIUM"),
            n_note=sum(1 for h in hits if h["sev"] == "NOTE"),
            unspecified_stereo=bool(st),
            alerts="; ".join(h["name"] for h in hits if h["sev"] == "HIGH")))

    df = pd.DataFrame(rows)
    n = len(df)
    n_valid = int(df["valid"].sum()) if n else 0
    clean = int(((df["valid"]) & (df["n_high"] == 0)).sum()) if n else 0
    summary = dict(n=n, n_valid=n_valid,
                   validity_pct=100.0 * n_valid / n if n else 0.0,
                   n_clean=clean, clean_pct=100.0 * clean / n if n else 0.0,
                   failure_modes=dict(sorted(fail_kinds.items(),
                                             key=lambda kv: -kv[1])),
                   alert_counts=dict(sorted(alert_counts.items(),
                                            key=lambda kv: -kv[1])))

    if show:
        h = ["<div style='font-family:system-ui,sans-serif'>",
             "<h3 style='color:#2c5c40;margin:4px 0'>Batch report</h3>",
             f"<p><b>{n}</b> structures &nbsp;&middot;&nbsp; "
             f"<b>{summary['validity_pct']:.1f}%</b> valid "
             f"({n_valid}/{n}) &nbsp;&middot;&nbsp; "
             f"<b>{summary['clean_pct']:.1f}%</b> valid with no high-severity "
             f"alert ({clean}/{n})</p>"]
        if fail_kinds:
            h.append("<b>Why the invalid ones failed</b><ul>")
            for k, c in summary["failure_modes"].items():
                h.append(f"<li>{k}: {c}</li>")
            h.append("</ul>")
        if alert_counts:
            h.append(f"<b>Most frequent alerts among the valid molecules</b>"
                     f"<ul>")
            for k, c in list(summary["alert_counts"].items())[:top]:
                h.append(f"<li>{k}: {c}</li>")
            h.append("</ul>")
        h.append("</div>")
        display(HTML("".join(h)))
        display(df.head(25))
    return df, summary


# ======================================================================
#  COUNTEREXAMPLES - alerts are heuristics, not verdicts
# ======================================================================

APPROVED = [
    ("metformin",      "CN(C)C(=N)NC(=N)N",
     "first-line type 2 diabetes drug; the guanidine is the pharmacophore"),
    ("acetaminophen",  "CC(=O)Nc1ccc(O)cc1",
     "the aniline and phenol are both flagged, and it is one of the most "
     "widely used drugs in the world"),
    ("aspirin",        "CC(=O)Oc1ccccc1C(=O)O",
     "an acetyl ester that is deliberately hydrolysed and deliberately "
     "acetylates COX"),
    ("ibuprofen",      "CC(C)Cc1ccc(cc1)C(C)C(=O)O",
     "clean, as a control"),
    ("afatinib",       "CN(C)C/C=C/C(=O)Nc1cc2c(Nc3ccc(F)c(Cl)c3)ncnc2cc1OC1CCOC1",
     "the Michael acceptor is the point: it is a covalent EGFR inhibitor"),
    ("bortezomib",     "CC(C)C[C@@H](NC(=O)[C@@H](Cc1ccccc1)NC(=O)c1cnccn1)B(O)O",
     "a boronic acid warhead, approved for multiple myeloma"),
    ("penicillin G",   "CC1(C)S[C@@H]2[C@H](NC(=O)Cc3ccccc3)C(=O)N2[C@H]1C(=O)O",
     "the strained beta-lactam is the mechanism"),
    ("melphalan",      "N[C@@H](Cc1ccc(cc1)N(CCCl)CCCl)C(=O)O",
     "a nitrogen mustard alkylator, still used in multiple myeloma"),
    ("telaprevir (fragment)", "CCC(NC(=O)C1CC2CCCCC2CN1)C(=O)C(=O)NC1CC1",
     "the alpha-ketoamide IS the mechanism: the protease's catalytic residue "
     "adds to that ketone to give a stable hemiketal"),
    ("metronidazole",  "Cc1ncc([N+](=O)[O-])n1CCO",
     "a nitroimidazole that is Ames-positive, and has been front-line "
     "therapy for anaerobic and protozoal infection for decades"),
    ("dapsone",        "Nc1ccc(cc1)S(=O)(=O)c1ccc(N)cc1",
     "two aromatic amines; still on the WHO essential medicines list for "
     "leprosy"),
    ("isoniazid",      "NNC(=O)c1ccncc1",
     "a hydrazide; front-line tuberculosis therapy since 1952"),
    ("dacarbazine",    "CN(C)N=Nc1[nH]cnc1C(=O)N",
     "an azo-triazene prodrug for melanoma"),
]


def counterexamples(show=True):
    """Run approved drugs through the same filters. Expect false positives."""
    rows = []
    for nm, smi, note in APPROVED:
        v = check_validity(smi)
        if not v["ok"]:
            rows.append(dict(drug=nm, valid=False, n_high=0, flags="(parse)",
                             why_it_is_still_a_drug=note))
            continue
        hits = scan_alerts(v["mol"])
        high = [h["name"] for h in hits if h["sev"] == "HIGH"]
        rest = [h["name"] for h in hits if h["sev"] != "HIGH"]
        rows.append(dict(drug=nm, valid=True, n_high=len(high),
                         n_alerts=len(hits),
                         flags="; ".join(high + rest)[:200] or "none",
                         why_it_is_still_a_drug=note))
    df = pd.DataFrame(rows)
    if show:
        n_flag = int((df["flags"] != "none").sum())
        display(HTML(
            "<div style='font-family:system-ui,sans-serif'>"
            "<h3 style='color:#d68910;margin:4px 0'>Reality check &mdash; "
            "approved drugs through the same filter</h3>"
            f"<p><b>{n_flag} of {len(df)}</b> of these marketed drugs trip at "
            "least one alert. Every one of them is or has been on the market. "
            "An alert is a reason to think and to justify, not a reason to "
            "delete a molecule. Where the flagged group <i>is</i> the "
            "mechanism &mdash; the beta-lactam in penicillin, the Michael "
            "acceptor in afatinib, the boronic acid in bortezomib &mdash; "
            "removing it would remove the drug.</p></div>"))
        display(df)
    return df


# ======================================================================
#  OUTPUT
# ======================================================================

def save_report(res, filename="structure_check.html", author=""):
    """Self-contained HTML report for one molecule - open or print to PDF."""
    mol = res.get("mol")
    if mol is None:
        svg = ""
    elif res["valid"]:
        svg = draw_mol(mol,
                       highlight=sorted({i for h in res["alerts"]
                                         for i in h["atoms"]}),
                       size=(500, 400), kekulize=True)
    else:
        bad = sorted({i for p in res["problems"]
                      for i in p.get("highlight", p["atoms"])})
        svg = draw_mol(mol, highlight=bad,
                       notes=problem_notes(mol, res["problems"]),
                       atom_colours={i: SEV_RGB["HIGH"] for i in bad},
                       size=(500, 400), kekulize=False)
    parts = [f"<!doctype html><meta charset='utf-8'>"
             f"<title>Structure check - {res['name']}</title>"
             f"<body style='font-family:system-ui,sans-serif;max-width:820px;"
             f"margin:30px auto;line-height:1.5'>"
             f"<h1 style='color:#2c5c40'>Structure sanity check</h1>"
             f"<p><b>{res['name']}</b><br><code>{res['smiles']}</code>"
             f"{'<br>' + author if author else ''}</p>{svg}"]
    if not res["valid"]:
        parts.append("<h2 style='color:#8b0000'>Stage 1 &mdash; not a valid "
                     "molecule</h2>")
        for p in res["problems"]:
            parts.append(f"<h3>{p['kind']}</h3><p>{p['plain']}</p>"
                         f"<p><b>Fix:</b> {p['fix']}</p>")
    else:
        parts.append("<h2 style='color:#2c5c40'>Stage 1 &mdash; valid</h2>")
        if not res["alerts"]:
            parts.append("<h2>Stage 2 &mdash; no alerts</h2>")
        else:
            parts.append(f"<h2>Stage 2 &mdash; {len(res['alerts'])} alert(s)"
                         f"</h2>")
            for h in res["alerts"]:
                c = SEV_COLOUR.get(h["sev"], "#333")
                parts.append(
                    f"<h3 style='color:{c}'>[{h['sev']}] {h['name']}</h3>"
                    f"<p style='color:#555;font-size:13px'>{h['cat']} &middot; "
                    f"atoms {' '.join(str(i+1) for i in h['atoms'][:12])}</p>"
                    f"<p>{h['why']}</p>"
                    + (f"<p><b>Suggested fix:</b> {h['fix']}</p>"
                       if h["fix"] else ""))
    seen = res.get("considered") or []
    if seen:
        parts.append("<h2 style='color:#5a605c'>Considered and not flagged</h2>"
                     "<p style='font-size:13px;color:#666'>Present in the "
                     "molecule, looked at, and deliberately not treated as "
                     "alerts.</p><ul>")
        for c in seen:
            times = f" &times;{c['n']}" if c["n"] > 1 else ""
            parts.append(f"<li><b>{c['name']}</b>{times} &mdash; {c['note']}</li>")
        parts.append("</ul>")
    for h in res.get("stereo", []):
        parts.append(
            f"<h3 style='color:#2e6ea6'>[INCOMPLETE] {h['name']}</h3>"
            f"<p>{h['why']}</p><p><b>What to do:</b> {h['fix']}</p>")
    parts.append(f"<hr><p style='font-size:11px;color:#999'>Structure "
                 f"Validation and Alerts v{VERSION} ({VERSION_DATE}) &mdash; "
                 f"{TOOL_AUTHOR}. "
                 f"<a href='{TOOL_URL}'>{TOOL_URL}</a></p>")
    parts.append(
        "<hr><p style='font-size:12px;color:#777'><b>This report can be "
        "wrong, in both directions.</b> It matches patterns, and a pattern "
        "has no judgement: it fires on a substructure whether or not the "
        "surrounding chemistry makes it a real problem, and it stays silent "
        "on anything nobody wrote a pattern for. A clean result means nothing "
        "matched &mdash; not that nothing is wrong. Many approved drugs trip "
        "these alerts. Use this to justify a design decision, not to settle "
        "one, and have a chemist read it.</p></body>")
    with open(filename, "w") as fh:
        fh.write("".join(parts))
    print(f"  wrote {filename}")
    return filename


def download(filename=None):
    """Download a file from Colab. Does nothing outside Colab."""
    try:
        from google.colab import files
    except Exception:
        print("  not running in Colab - find your files in the file browser")
        return
    targets = [filename] if filename else [f for f in os.listdir(".")
                                           if f.endswith((".html", ".csv",
                                                          ".sdf", ".svg"))]
    for t in targets:
        if os.path.exists(t):
            files.download(t)


print(f"  Structure Validation and Alerts  v{VERSION}  ({VERSION_DATE})")
print(f"  ready - {len(ALERTS)} custom patterns + PAINS + Brenk + NIH catalogs")
print( "  if you report a problem, quote the version above")


def draw_mol_png(mol, highlight=(), atom_colours=None, notes=None,
                 labels=None, size=(620, 440), kekulize=True, radius=0.32):
    """PNG bytes via RDKit's bundled Cairo drawer - no extra system libs."""
    if mol is None:
        return None
    m = Chem.Mol(mol)
    for i, txt in (notes or {}).items():
        if i < m.GetNumAtoms():
            m.GetAtomWithIdx(int(i)).SetProp("atomNote", txt)
    for i, txt in (labels or {}).items():
        if i < m.GetNumAtoms():
            m.GetAtomWithIdx(int(i)).SetProp("atomLabel", txt)
    try:
        AllChem.Compute2DCoords(m)
    except Exception:
        pass
    d = rdMolDraw2D.MolDraw2DCairo(size[0], size[1])
    o = d.drawOptions()
    o.highlightRadius = radius
    o.annotationFontScale = 0.75
    hl = [int(i) for i in highlight if i < m.GetNumAtoms()]
    cols = {i: (atom_colours or {}).get(i, SEV_RGB["HIGH"]) for i in hl}
    try:
        rdMolDraw2D.PrepareAndDrawMolecule(d, m, highlightAtoms=hl,
                                           highlightAtomColors=cols,
                                           kekulize=kekulize)
    except Exception:
        try:
            rdMolDraw2D.PrepareAndDrawMolecule(d, m, highlightAtoms=hl,
                                               highlightAtomColors=cols,
                                               kekulize=False)
        except Exception:
            d.DrawMolecule(m)
    d.FinishDrawing()
    return d.GetDrawingText()


def save_pdf(res, filename="structure_check.pdf", author=""):
    """
    A real PDF, for handing in. Needs reportlab (pure python):
        !pip install reportlab
    Falls back to the HTML report if reportlab is missing.
    """
    try:
        from reportlab.lib.pagesizes import A4
        from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
        from reportlab.lib.units import mm as MM
        from reportlab.lib import colors as RC
        from reportlab.platypus import (SimpleDocTemplate, Paragraph, Spacer,
                                        Image, Table, TableStyle)
        from reportlab.lib.utils import ImageReader
    except ImportError:
        print("  reportlab is not installed - run:  !pip install reportlab")
        print("  writing the HTML report instead")
        return save_report(res, filename.replace(".pdf", ".html"), author)

    import io
    ss = getSampleStyleSheet()
    body = ParagraphStyle("body", parent=ss["BodyText"], fontSize=9.5,
                          leading=13)
    small = ParagraphStyle("small", parent=body, fontSize=8,
                           textColor=RC.HexColor("#666666"))
    h1 = ParagraphStyle("h1", parent=ss["Heading1"], fontSize=17,
                        textColor=RC.HexColor("#2c5c40"))

    story, mol = [], res.get("mol")
    story.append(Paragraph("Structure sanity check", h1))
    story.append(Paragraph(f"<b>{res['name']}</b>", body))
    story.append(Paragraph(f"<font face='Courier'>{res['smiles']}</font>",
                           body))
    if author:
        story.append(Paragraph(author, small))
    story.append(Spacer(1, 5 * MM))

    colour, title, blurb = VERDICT[res["verdict"]]
    t = Table([[Paragraph(f"<b><font color='white' size='13'>{title}</font>"
                          f"</b><br/><font color='white' size='8'>{blurb}"
                          f"</font>", body)]],
              colWidths=[165 * MM])
    t.setStyle(TableStyle([("BACKGROUND", (0, 0), (-1, -1),
                            RC.HexColor(colour)),
                           ("LEFTPADDING", (0, 0), (-1, -1), 8),
                           ("RIGHTPADDING", (0, 0), (-1, -1), 8),
                           ("TOPPADDING", (0, 0), (-1, -1), 7),
                           ("BOTTOMPADDING", (0, 0), (-1, -1), 7)]))
    story += [t, Spacer(1, 5 * MM)]

    if mol is not None:
        if res["valid"]:
            worst = {}
            for h in res["alerts"]:
                for i in h["atoms"]:
                    if i not in worst or SEV_ORDER[h["sev"]] < \
                       SEV_ORDER[worst[i]]:
                        worst[i] = h["sev"]
            png = draw_mol_png(mol, highlight=sorted(worst),
                               atom_colours={i: SEV_RGB[s]
                                             for i, s in worst.items()})
        else:
            bad = sorted({i for p in res["problems"]
                          for i in p.get("highlight", p["atoms"])})
            png = draw_mol_png(mol, highlight=bad,
                               labels=problem_notes(mol, res["problems"]),
                               atom_colours={i: PALE_MARK for i in bad},
                               radius=0.40, kekulize=False)
        if png:
            story.append(Image(io.BytesIO(png), width=124 * MM,
                               height=88 * MM))
            story.append(Spacer(1, 4 * MM))

    if not res["valid"]:
        for p in res["problems"]:
            story.append(Paragraph(f"<b>{p['kind']}</b>", body))
            story.append(Paragraph(p["plain"], body))
            story.append(Paragraph(f"<b>Fix:</b> {p['fix']}", body))
            story.append(Spacer(1, 3 * MM))
    else:
        for h in res["alerts"]:
            c = SEV_COLOUR.get(h["sev"], "#333333")
            pos = " ".join(str(i + 1) for i in h["atoms"][:12]) or "-"
            story.append(Paragraph(
                f"<font color='{c}'><b>[{h['sev']}] {h['name']}</b></font>",
                body))
            story.append(Paragraph(f"{h['cat']} &middot; atoms {pos}", small))
            story.append(Paragraph(h["why"], body))
            if h["fix"]:
                story.append(Paragraph(f"<b>Suggested fix:</b> {h['fix']}",
                                       body))
            story.append(Spacer(1, 3 * MM))
        for h in res.get("stereo", []):
            story.append(Paragraph(
                f"<font color='#2e6ea6'><b>[INCOMPLETE] {h['name']}</b>"
                f"</font>", body))
            story.append(Paragraph(h["why"], body))
            story.append(Paragraph(f"<b>What to do:</b> {h['fix']}", body))
            story.append(Spacer(1, 3 * MM))
        seen = res.get("considered") or []
        if seen:
            story.append(Spacer(1, 2 * MM))
            story.append(Paragraph(
                "<font color='#5a605c'><b>Considered and not flagged</b>"
                "</font>", body))
            for c in seen:
                times = f" &times;{c['n']}" if c["n"] > 1 else ""
                story.append(Paragraph(f"<b>{c['name']}</b>{times} &mdash; "
                                       f"{c['note']}", small))
                story.append(Spacer(1, 1.5 * MM))

    story.append(Spacer(1, 5 * MM))
    story.append(Paragraph(
        "<b>This report can be wrong, in both directions.</b> It matches "
        "patterns, and a pattern has no judgement: it fires on a "
        "substructure whether or not the surrounding chemistry makes it a "
        "real problem, and it stays silent on anything nobody wrote a "
        "pattern for. A clean result means nothing matched &mdash; not that "
        "nothing is wrong. Many approved drugs trip these alerts. Use this "
        "to justify a design decision, not to settle one, and have a chemist "
        "read it.", small))
    story.append(Spacer(1, 2 * MM))
    story.append(Paragraph(
        f"Generated by Structure Validation and Alerts v{VERSION} "
        f"({VERSION_DATE}) &mdash; {TOOL_AUTHOR}. {TOOL_URL}", small))

    SimpleDocTemplate(filename, pagesize=A4,
                      leftMargin=22 * MM, rightMargin=22 * MM,
                      topMargin=18 * MM, bottomMargin=18 * MM).build(story)
    print(f"  wrote {filename}")
    return filename


# ----------------------------------------------------------------------
#  DOWNLOAD BUTTONS
# ----------------------------------------------------------------------
#  google.colab.files.download() fires a save the instant the cell runs,
#  which means the file lands in Downloads whether you wanted it or not,
#  and asking for two files trips the browser's "allow multiple
#  downloads?" prompt. A plain HTML link with the file embedded as a data
#  URI is better on every count: nothing is saved until it is clicked,
#  there is no popup to allow, and it works in Jupyter and VS Code as
#  well as Colab.

_MIME = {".pdf": "application/pdf", ".html": "text/html",
         ".csv": "text/csv", ".sdf": "chemical/x-mdl-sdfile",
         ".svg": "image/svg+xml", ".png": "image/png",
         ".txt": "text/plain"}


def download_button(filename, label=None, colour="#2c5c40"):
    """A real button. Nothing is saved until the student clicks it."""
    import base64, os
    if not os.path.exists(filename):
        print(f"  {filename} does not exist yet")
        return
    ext = os.path.splitext(filename)[1].lower()
    mime = _MIME.get(ext, "application/octet-stream")
    kb = max(1, round(os.path.getsize(filename) / 1024))
    with open(filename, "rb") as fh:
        b64 = base64.b64encode(fh.read()).decode()
    label = label or f"Download {filename}"
    display(HTML(
        f"<a download='{filename}' href='data:{mime};base64,{b64}' "
        f"style='display:inline-block;background:{colour};color:#fff;"
        f"padding:9px 18px;margin:5px 8px 5px 0;border-radius:6px;"
        f"text-decoration:none;font-family:system-ui,sans-serif;"
        f"font-size:14px;font-weight:600'>&#11015;&nbsp; {label}"
        f"<span style='opacity:.75;font-weight:400'> &nbsp;{kb} KB</span>"
        f"</a>"))


def download_buttons(*filenames):
    """Buttons for several files, side by side."""
    import os
    shown = [f for f in filenames if f and os.path.exists(f)]
    if not shown:
        print("  nothing to download yet - run the report cell first")
        return
    for f in shown:
        ext = os.path.splitext(f)[1].lower()
        download_button(f, label=f"Download {ext.lstrip('.').upper()}",
                        colour="#2c5c40" if ext == ".pdf" else "#4a6fa5")

<div style="background:#2c5c40; color:#ffffff; padding:12px 22px; border-radius:8px;
            margin-top:28px;">
<h2 style="color:#ffffff; margin:0; font-size:30px;">&#9733; STEP 2 &mdash; Check your molecule</h2>
</div>

<span style="background:#d68910; color:#ffffff; padding:2px 10px; border-radius:4px; font-size:13px; font-weight:bold;">YOU EDIT THIS ONE</span>

Paste your SMILES into the box. Leave it blank and the cell runs two worked examples
instead &mdash; one that fails Stage 1 and one that passes Stage 1 and then trips Stage 2 &mdash;
so you can see what each kind of report looks like.

In [ ]:
# ===== CELL 3 ===== EDIT THIS LINE =====
SMILES = ""        # your molecule, e.g. "c1ccccc1C=O"
NAME   = ""        # optional label for the report
# =======================================================

if not SMILES.strip():
    print("SMILES is empty, so these are the two worked examples.")
    print("Put your own molecule in the box above and re-run.\n")

    print("=" * 66)
    print("EXAMPLE 1 - a CH2 sitting in a ring drawn as aromatic")
    print("=" * 66)
    res = check("c1ccCcc1", name="aromatic ring with a CH2 in it")

    print("\n" + "=" * 66)
    print("EXAMPLE 2 - a valid molecule that trips Stage 2")
    print("=" * 66)
    res = check("c1ccccc1C=O", name="benzaldehyde")
else:
    res = check(SMILES, name=NAME)

### Reading the report

Every molecule gets one of four banners:

| Banner | Meaning |
|---|---|
| **dark red** &mdash; not a valid molecule | Stage 1 failed. The most severe outcome: this is not a chemistry problem but a drawing problem, and nothing downstream can run. |
| **red** &mdash; high-severity alert | Something will not survive as drawn, or is a hard covalent liability. |
| **amber** &mdash; worth justifying | No high-severity problem, but be ready to say why this group is here. |
| **green** &mdash; no alerts matched | Nothing matched a pattern this notebook knows about. **This is the absence of a flag, not evidence of a good molecule.** It says nothing about potency, selectivity, metabolism, or whether anyone can make it. |

Only the atoms that **are** the alert are coloured. A SMARTS pattern often needs a context
atom to match correctly &mdash; the aldehyde pattern is anchored to the carbon next door so it
does not also match formic acid &mdash; but that neighbour is not part of the problem and is
left uncoloured.

A single group is reported **once**. If a curated pattern and the published catalogues all
flag the same aldehyde, you get one row that notes it was *also flagged by Brenk, NIH*,
rather than three rows saying the same thing.

In the drawing, each flagged atom is coloured by the **worst** severity that hit it:
<span style="color:#c0392b"><b>red = high</b></span>,
<span style="color:#d68910"><b>amber = medium</b></span>, grey = note. A molecule with an
aldehyde and a catechol shows both colours at once, which is the point &mdash; one colour per
molecule would hide that.

**If Stage 1 fails** you get the problem type, the specific atom, a plain-language
explanation and a fix. Atom numbers in the report are 1-based and match the circled atom
in the drawing. Stage 2 does not run, and that is not the notebook being unhelpful &mdash;
there is no molecule yet to check.

| Stage 1 problem | What it means |
|---|---|
| `SYNTAX` | The text is not readable as SMILES. Usually an unclosed ring digit or an unbalanced bracket. |
| `VALENCE` | An atom has more bonds than the element allows. Carbon with five bonds is the classic. |
| `KEKULIZE` | The ring is drawn as aromatic but no alternating single/double pattern can be written. Usually a CH<sub>2</sub> or another saturated atom in the ring. |

The offending atom is marked in red and labelled with what it actually is &mdash; `CH2` &mdash;
so you can see *what* is wrong there, not just *where*. Its number is given in the caption
under the structure rather than on the atom, because an `=` sign next to a structure reads
as a double bond. Carbons are drawn as bare
vertices by default, so without that note one ring carbon looks exactly like the next five.
Only the flagged atom is numbered; numbering every atom is readable on a benzene and
unreadable on anything drug-sized.

**If Stage 1 passes**, every Stage 2 hit carries a severity:

| Severity | Meaning |
|---|---|
| **HIGH** | Will not survive as drawn, or is a hard covalent liability. Fix it unless you meant it. |
| **MEDIUM** | A real alert. Defensible, but you should be able to say why. |
| **NOTE** | Context dependent. Shown so you think about it, not so you remove it. |

And a category: `UNSTABLE` (falls apart), `REACTIVE` (electrophile), `ASSAY` (interferes
with the readout), `MUTAGENIC` (a toxicophore with a known activation route to a DNA-reactive
species), `STRAIN` (geometrically impossible or nearly so), `INCOMPLETE` (valid, but the
drawing does not define a single compound).

<div style="border-left:5px solid #8b0000; background:#fdf5f5; padding:10px 16px; margin:12px 0;">
<b style="color:#8b0000">A caution specific to the MUTAGENIC category</b><br>
A mutagenicity alert is a different <i>kind</i> of claim from the rest of Stage 2.
"This hemiacetal hydrolyses" is chemistry you can predict from structure. "This is mutagenic"
is a prediction about the outcome of an assay in a living system, and these rule bases carry
substantial false-positive rates. <b>Metronidazole is Ames-positive and has been front-line
therapy for anaerobic infection for decades.</b> Dapsone carries two aromatic amines and is on
the WHO essential medicines list. Real assessment runs through Toxtree's Benigni&ndash;Bossa
rulebase, VEGA, or a commercial system under ICH&nbsp;M7 &mdash; not through this notebook.
Treat a <code>MUTAGENIC</code> flag as a prompt to read, not as a result.
</div>

Where a hit names a **Bioisostere Explorer** lookup, that is the group to search in the
other notebook to find what to put there instead.

<div style="background:#2c5c40; color:#ffffff; padding:12px 22px; border-radius:8px;
            margin-top:28px;">
<h2 style="color:#ffffff; margin:0; font-size:30px;">&#9733; STEP 3 &mdash; Reality check</h2>
</div>

**Run this before you trust any alert.** Ten drugs that are or have been
on the market, through the same filters your molecule just went through.

In [ ]:
# ===== CELL 4 =====
drugs = counterexamples()

Notice what happened. Metformin's guanidine, acetaminophen's phenol,
isoniazid's hydrazide, dacarbazine's azo group &mdash; all flagged, all on the market. And in
three cases the flagged group is not a liability that was tolerated, it is the **mechanism**:

- **afatinib** &mdash; the Michael acceptor is how it binds EGFR covalently
- **penicillin G** &mdash; the strained &beta;-lactam is what acylates the transpeptidase
- **bortezomib** &mdash; the boronic acid is the warhead that engages the proteasome
- **telaprevir** &mdash; the &alpha;-ketoamide is attacked by the protease's catalytic residue

And two that are flagged `MUTAGENIC` and were approved anyway:

- **metronidazole** &mdash; a nitroimidazole, Ames-positive, decades of front-line use
- **dapsone** &mdash; two aromatic amines, WHO essential medicines list

Remove the alert from any of those four and you have removed the drug. This is the whole
reason the report says *suggested fix* rather than *required fix*. The question an alert
asks you is **"can you justify this?"**, not **"will you delete this?"**

<div style="background:#2c5c40; color:#ffffff; padding:12px 22px; border-radius:8px;
            margin-top:28px;">
<h2 style="color:#ffffff; margin:0; font-size:30px;">&#9733; STEP 4 &mdash; Many molecules at once</h2>
</div>

<span style="background:#d68910; color:#ffffff; padding:2px 10px; border-radius:4px; font-size:13px; font-weight:bold;">YOU EDIT THIS ONE</span>

This is the research half. Point it at a list of SMILES &mdash; typically the output of a
generative model, an enumerated library, or a set of student submissions &mdash; and it
reports the validity rate **and the breakdown of why the failures failed**, which a single
"87% valid" number does not tell you.

You can pass a Python list, or the name of a text file with one SMILES per line.

In [ ]:
# ===== CELL 5 ===== EDIT THIS LINE =====
MOLECULES = []     # a list of SMILES, or a filename like "generated.smi"
# =========================================================

if not MOLECULES:
    print("MOLECULES is empty, so this is a worked example standing in for")
    print("the output of a generative model.\n")
    MOLECULES = [
        "c1ccccc1C=O",                    # valid, aldehyde
        "CC(C)(C)(C)C",                   # pentavalent carbon
        "c1ccCcc1",                       # CH2 in an aromatic ring
        "CCO",                            # valid, clean
        "OC(O)C",                         # gem-diol
        "C1CCCCC",                         # unclosed ring
        "CC(=O)Cl",                       # acyl chloride
        "CC(C)Cc1ccc(cc1)C(C)C(=O)O",     # ibuprofen, clean
        "c1ccncc1",                       # pyridine, clean
        "COOC",                           # peroxide
        "OC=Cc1ccccc1",                   # enol
        "C1OC1Cc1ccccc1",                 # epoxide
    ]

batch_df, batch_summary = check_batch(MOLECULES)

The two numbers that matter are **validity** (did it parse and sanitise)
and **clean** (valid *and* no high-severity alert). They are different questions and a
generative model can score well on one and badly on the other.

`batch_df` is a pandas DataFrame, so you can sort, filter and export it:

```python
batch_df.to_csv("batch_report.csv", index=False)
batch_df[batch_df.n_high > 0]            # only the problem molecules
batch_df[~batch_df.valid].failure        # why the invalid ones failed
```

<div style="background:#2c5c40; color:#ffffff; padding:12px 22px; border-radius:8px;
            margin-top:28px;">
<h2 style="color:#ffffff; margin:0; font-size:30px;">&#9733; STEP 5 &mdash; Keep the result</h2>
</div>

Writes a report for whatever you checked in Cell 3, in two formats.

**PDF** &mdash; `structure_check.pdf`. Use this for submitting work. It carries the verdict
banner, the structure with its colour coding, every alert with its reasoning, and the
considered-and-not-flagged list. Needs `reportlab`, which Cell 1 installs; if it is missing
the cell falls back to HTML rather than failing.

**HTML** &mdash; `structure_check.html`. The same content as a web page, which is easier to
read on screen and to copy text out of.

Running the cell writes both files and shows **two buttons**. Nothing reaches your computer
until you click one, so you can re-run this cell as often as you like without filling your
Downloads folder.

Both files are also written into Colab's own file browser &mdash; the folder icon in the left
sidebar. The buttons work the same way in Jupyter and VS Code, not just Colab.

If you would rather a file save itself the moment the cell runs,
`download("structure_check.pdf")` still does that.

In [ ]:
# ===== CELL 6 =====
AUTHOR = "your name here"
# =======================================================

save_pdf(res,    "structure_check.pdf",  author=AUTHOR)
save_report(res, "structure_check.html", author=AUTHOR)

# Nothing is saved to your computer until you click a button.
download_buttons("structure_check.pdf", "structure_check.html")

### Add your own alert &nbsp; <span style="background:#d68910; color:#ffffff; padding:2px 10px; border-radius:4px; font-size:13px; font-weight:bold;">YOU EDIT THIS ONE</span>

The 31 built-in patterns are a starting point, not a complete list. Add the ones your own
project cares about. Patterns are
[SMARTS](https://www.daylight.com/dayhtml/doc/theory/theory.smarts.html); you can test one
quickly with `Chem.MolFromSmarts(...)` and `mol.HasSubstructMatch(...)` before you add it.

In [ ]:
# ===== CELL 7 =====
add_alert(
    name   = "thioester",
    smarts = "[CX3](=[OX1])[SX2][#6]",
    cat    = "UNSTABLE",          # UNSTABLE / REACTIVE / STRAIN
    sev    = "MEDIUM",            # HIGH / MEDIUM / NOTE
    why    = "more electrophilic than the ester; hydrolyses and transfers the "
             "acyl group to thiols and amines, which is exactly what acetyl-CoA "
             "does in vivo",
    fix    = "use the ester or the amide",
    iso    = "ester",             # group to look up in Bioisostere Explorer, or ""
)

# re-run your molecule with the new pattern active
# res = check(SMILES, name=NAME)

---

## Exercises

**1. Break a molecule on purpose.** Take benzene, `c1ccccc1`, and make one ring carbon a
CH<sub>2</sub> so you get `c1ccCcc1`. Run Cell 3. Now fix it two different ways: once as
benzene, once as cyclohexane. Which atom did the report point at, and why was it not the
atom RDKit's raw error message listed?

**2. Five bonds.** Write a SMILES for a carbon with five bonds and run it. Then write the
same thing as a charged species that *is* legal, and explain the difference.

**3. The aldehyde problem.** Run `c1ccccc1C=O`. Then open the Bioisostere Explorer, look up
`aldehyde`, and pick a replacement. Bring the replacement back here and check it. Did you
trade one alert for another?

**4. Tautomers.** Run `OC=Cc1ccccc1`, which is drawn as an enol. Draw the keto form yourself
and check it. Both are "valid" to RDKit &mdash; what did Stage 2 tell you that Stage 1 could not?

**5. Which ring is real.** Run these three and explain the different outcomes:
`c1cccc1`, `c1ccccc1`, `C1CCCCC1`.

**5b. Why is the ketone not flagged?** Run `O=C(C)C1CCC(CC2=CC=CC=C2)CC1` and read the grey
**considered and not flagged** box. The methyl ketone passes without an alert. Now run the aldehyde version, `O=CC1CCC(CC2=CC=CC=C2)CC1`, and the
&alpha;,&beta;-unsaturated version, `O=C(/C=C/C)C1CCC(CC2=CC=CC=C2)CC1`. Same carbonyl, three
different verdicts. Explain the difference using the hydration equilibrium: in water
formaldehyde is more than 99.9% hydrate, acetaldehyde roughly half, acetone a few tenths of
a percent. Then run `CCCCCCCCC(=O)C(F)(F)F` and the telaprevir fragment from Cell 4, and say
what the substituents did to that same carbonyl.

**5c. The molecule is valid and still wrong.** Run `O=C(C)C1CCC(CC2=CC=CC=C2)CC1` again and
read the blue `INCOMPLETE` panel. How many compounds does that drawing actually describe?
Draw the one you meant, and check that the panel goes away.

**5d. Follow one activation route.** Run these four in order: `Nc1ccccc1` (aniline),
`ONc1ccccc1` (the N-hydroxy compound), `O=Nc1ccccc1` (the nitroso), and
`O=[N+]([O-])c1ccccc1` (nitrobenzene). Read the `MUTAGENIC` explanations and draw the
metabolic scheme that connects all four. Which two are the *ultimate* mutagen, and which two
are precursors that need enzymes first? Then explain why a nitro group and an aromatic amine
&mdash; chemically opposite &mdash; end up at the same place.

**6. Justify an alert.** Pick any approved drug from Cell 4 that was flagged. In two
sentences, argue why it reached the market anyway. Then pick one where you think the alert
*should* have stopped it, and say why it did not.

**7. Add a pattern.** Use Cell 7 to add an alert for a group your own project cares about.
Write the `why` field as though you were explaining it to next year's class &mdash; that field
is the part that teaches.

**8. Audit a model.** If you have generated molecules from any model, run them through
Cell 5. Report the validity percentage, the clean percentage, and the single most common
failure mode. Then answer the real question: were the invalid ones invalid for *chemical*
reasons or for *string* reasons?

---

<div style="border:2px solid #3f7d58; border-radius:10px; padding:4px 22px 18px 22px;
            background:#f4faf6; color:#1b1b1b; font-size:14px;">

<h3 style="color:#2c5c40; margin-bottom:2px;"><b>Before you trust this</b></h3>

<p><b>This notebook can be wrong, in both directions.</b> Stage 2 matches patterns, and
a pattern has no judgement: it fires on a substructure whether or not the surrounding
chemistry makes it a real problem, and it stays silent on anything nobody wrote a
pattern for. <b>A green verdict means nothing matched &mdash; not that nothing is
wrong.</b> Treat every result as a reason to think, not an answer to accept.</p>

<h3 style="color:#2c5c40; margin-bottom:2px; margin-top:16px;"><b>What it does not do</b></h3>

<p><b>It does not predict metabolism.</b> Metabolic soft spots &mdash; sites of oxidation,
glucuronidation, ester hydrolysis in plasma &mdash; are a separate problem and are not
checked here.</p>

<p><b>It does not predict whether your molecule binds anything.</b> A molecule can pass both
stages cleanly and be completely inactive.</p>

<p><b>It does not know about your synthesis.</b> A structure can be valid, stable, and
unflagged, and still take fifteen steps nobody will run.</p>

<p><b>Stage 2 is pattern matching, not chemistry.</b> SMARTS patterns have no concept of
context. A pattern matches or it does not, and the surrounding electronics that would
actually decide whether a group is reactive are invisible to it. Every hit needs a chemist
to read it. That is why the report gives you a reason rather than a verdict.</p>

<p><b>It does not predict mutagenicity.</b> The <code>MUTAGENIC</code> category reports that a
substructure has a known activation route to a DNA-reactive species. It does not run an Ames
test, and the published rule bases it draws on have substantial false-positive rates. For
real assessment use Toxtree (Benigni&ndash;Bossa), VEGA, or a commercial system under
ICH&nbsp;M7.</p>

<p><b>The alert lists are not complete and not neutral.</b> PAINS was derived from six
assays at one institution; Brenk's list was assembled for screening libraries against
neglected-disease targets. Both are useful and both carry the assumptions of where they
came from. Roughly 5% of approved drugs contain a PAINS motif.</p>

<h3 style="color:#2c5c40; margin-bottom:2px; margin-top:16px;"><b>Sources</b></h3>
<p style="font-size:13px;">
Brenk et&nbsp;al., <i>ChemMedChem</i> <b>2008</b>, 3, 435&ndash;444 &mdash; unwanted substructure list<br>
Baell &amp; Holloway, <i>J. Med. Chem.</i> <b>2010</b>, 53, 2719&ndash;2740 &mdash; PAINS filters<br>
RDKit <code>FilterCatalog</code> &mdash; PAINS, Brenk and NIH catalogues as implemented<br>
Kazius, McGuire &amp; Bursi, <i>J. Med. Chem.</i> <b>2005</b>, 48, 312&ndash;320 &mdash; mutagenicity toxicophores<br>
Benigni &amp; Bossa &mdash; structural alert rulebase, as implemented in Toxtree<br>
RDKit <code>DetectChemistryProblems</code> &mdash; the Stage 1 machinery<br>
TeachOpenCADD talktorial T003 &mdash; the library-filtering companion to this notebook
</p>

</div>

<br>